In [1]:
import os
from collections import defaultdict

# ── LABEL MAP ─────────────────────────────────────────────────────────────────
LABEL_NAMES = {
    "1": "Schizophrenia",
    "2": "ADHD",
    "3": "Healthy Control",
    "4": "Bipolar",
    "5": "Schizophrenia Spectrum",  # NUSDAST group 5
    "6": "Control Sibling",
}

def count_by_group(nifti_folders: list[str]):
    """
    Given a list of NIFTI folder paths, scan all .nii.gz files,
    extract the group number from the filename pattern:
        <subjectname>_T1_<grpno>_<datasetcode>.nii.gz
        or more generally any filename where second-to-last _ segment is grpno
    and return counts grouped by diagnosis label.

    Args:
        nifti_folders: list of folder paths to scan

    Returns:
        dict with group label -> count, plus per-dataset breakdown
    """

    group_counts    = defaultdict(int)   # grp -> total count
    dataset_counts  = defaultdict(lambda: defaultdict(int))  # dataset -> grp -> count
    unknown         = []                  # files that couldn't be parsed
    total           = 0

    for folder in nifti_folders:
        if not os.path.isdir(folder):
            print(f"  [WARN] Folder not found: {folder}")
            continue

        files = [f for f in os.listdir(folder) if f.endswith(".nii.gz")]

        for fname in files:
            # Remove .nii.gz, split by _
            base   = fname.replace(".nii.gz", "")
            parts  = base.split("_")

            # Need at least: <name> _ <grp> _ <dataset>
            if len(parts) < 3:
                unknown.append(fname)
                continue

            grp        = parts[-2]   # second to last segment
            dataset    = parts[-1]   # last segment

            if grp not in LABEL_NAMES:
                unknown.append(fname)
                continue

            group_counts[grp]              += 1
            dataset_counts[dataset][grp]   += 1
            total                          += 1

    # ── PRINT REPORT ──────────────────────────────────────────────────────────
    print("=" * 55)
    print(f"  TOTAL NIfTI FILES : {total}")
    print("=" * 55)
    print(f"  {'GROUP':<25} {'LABEL':<8} {'COUNT':>6}")
    print("-" * 55)
    for grp in sorted(group_counts.keys(), key=lambda x: int(x)):
        name  = LABEL_NAMES[grp]
        count = group_counts[grp]
        print(f"  {name:<25} {grp:<8} {count:>6}")
    print("=" * 55)

    print("\n  BREAKDOWN BY DATASET:")
    print("-" * 55)
    for ds in sorted(dataset_counts.keys(), key=lambda x: int(x) if x.isdigit() else x):
        ds_total = sum(dataset_counts[ds].values())
        print(f"\n  Dataset {ds}  ({ds_total} files)")
        for grp in sorted(dataset_counts[ds].keys(), key=lambda x: int(x)):
            name  = LABEL_NAMES.get(grp, f"Unknown({grp})")
            count = dataset_counts[ds][grp]
            print(f"    {name:<25} {grp:<8} {count:>6}")

    if unknown:
        print(f"\n  [WARN] {len(unknown)} files could not be parsed:")
        for f in unknown:
            print(f"    {f}")

    print()
    return dict(group_counts)


# ── USAGE ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    pathsE = [
    r"E:\MRI_data\open_neuro\open_neuro_0_71\scans\NIFTI",
    r"E:\MRI_data\open_neuro\open_neuro_1_26\scans\NIFTI",
    r"E:\MRI_data\open_neuro\open_neuro_2_16\scans\NIFTI",
    r"E:\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI",
    r"E:\MRI_data\open_neuro\open_neuro_4_50\scans\NIFTI",
    r"E:\MRI_data\schizconnect\cobre\scans\NIFTI",
    r"E:\MRI_data\NITRC\NUSDAST\scans\NIFTI",
  
    ]
    
    paths = [
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_0_71\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_1_26\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_16\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\cobre\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI",
  
    ]

    counts = count_by_group(paths)

  TOTAL NIfTI FILES : 876
  GROUP                     LABEL     COUNT
-------------------------------------------------------
  Schizophrenia             1           361
  Healthy Control           3           515

  BREAKDOWN BY DATASET:
-------------------------------------------------------

  Dataset 0  (71 files)
    Schizophrenia             1            71

  Dataset 1  (52 files)
    Schizophrenia             1            26
    Healthy Control           3            26

  Dataset 2  (16 files)
    Schizophrenia             1            16

  Dataset 3  (43 files)
    Schizophrenia             1            23
    Healthy Control           3            20

  Dataset 4  (175 files)
    Schizophrenia             1            50
    Healthy Control           3           125

  Dataset 6  (163 files)
    Schizophrenia             1            96
    Healthy Control           3            67

  Dataset 7  (356 files)
    Schizophrenia             1            79
    Healthy Control  

In [19]:
import os
import re
from collections import defaultdict

def count_nifti_suffixes(directory):
    
    if not os.path.isdir(directory):
        print(f"[ERROR] Directory not found: {directory}")
        return
    # Pattern to capture the last _digit_digit before .nii.gz
    pattern = re.compile(r'(_\d+_\d+)\.nii\.gz$')
    suffix_counts = defaultdict(int)
    unmatched     = []
    total         = 0
    for filename in os.listdir(directory):
        if not filename.endswith(".nii.gz"):
            continue
        total += 1
        match = pattern.search(filename)
        if match:
            suffix = match.group(1)   # e.g. "_1_2"
            suffix_counts[suffix] += 1
        else:
            unmatched.append(filename)

    # ── PRINT RESULTS ─────────────────────────────────────────────────────────
    print(f"\nDirectory : {directory}")
    print(f"Total .nii.gz files found: {total}")
    print("-" * 40)
    if suffix_counts:
        print(f"{'Suffix':<15} {'Count':>6}")
        print("-" * 40)
        for suffix, count in sorted(suffix_counts.items()):
            print(f"{suffix:<15} {count:>6}")
    else:
        print("No matching suffix patterns found.")
    if unmatched:
        print(f"\nFiles with no recognised suffix pattern ({len(unmatched)}):")
        for f in unmatched:
            print(f"  {f}")
    print("-" * 40)
    print(f"Total matched : {sum(suffix_counts.values())}")
    print(f"Total unmatched: {len(unmatched)}\n")

    return dict(suffix_counts)


# ── EXAMPLE USAGE ─────────────────────────────────────────────────────────────

paths = [
    # r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_0_71\scans\NIFTI",
    # r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_1_26\scans\NIFTI",
    # r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_16\scans\NIFTI",
    # r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI",
    # r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\cobre\scans\NIFTI",
    # r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI",
    # r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\olderNIFTI",
         ]

for path in paths:
    count_nifti_suffixes(path)


Directory : D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\cobre\scans\NIFTI
Total .nii.gz files found: 328
----------------------------------------
Suffix           Count
----------------------------------------
_1_7                79
_3_7               249
----------------------------------------
Total matched : 328
Total unmatched: 0



In [1]:
#code to count no folders in a directory
import os
def count_folders(directory):
    return len([name for name in os.listdir(directory) if os.path.isdir(os.path.join(directory, name))])
# dictory = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnectdupe"
dictory = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect"
dictory = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\controls"

print(count_folders(dictory))

668


In [2]:
print(os.listdir(dictory))

['1202146', '1216386', '1272786', '1527186', '1529966', '1552206', '1609486', '1611086', '1613626', '1714686', '1742206', '1868926', '1909286', '1909326', '1923485', '1923495', '1923496', '1923501', '1923502', '1923527', '1923528', '1923551', '1923552', '1923595', '1923596', '1923627', '1923628', '1923645', '1923646', '1923647', '1923648', '1923691', '1923692', '1923697', '1923698', '1923721', '1923722', '1923747', '1923748', '1923753', '1923754', '1923759', '1923760', '1923765', '1923766', '1923784', '1923785', '1923834', '1923835', '1924209', '1924210', '1924211', '1924212', '1924219', '1924220', '1924221', '1924227', '1924228', '1924229', '1924235', '1924236', '1924237', '1924238', '1924253', '1924254', '1924255', '1924291', '1924292', '1924293', '1924294', '1924336', '1924337', '1924338', '1924352', '1924353', '1924354', '1924360', '1924361', '1924362', '1924363', '1924369', '1924370', '1924371', '1924385', '1924386', '1924387', '1924393', '1924394', '1924395', '1924429', '1924430'

In [13]:
import pandas as pd
dat=pd.read_csv(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\schiz_strict.csv")
dat2=pd.read_csv(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\schiz_strict.csv")
dat.sort_values(by="Datauri",inplace=True)
dat.to_csv(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\schiz_strict.csv",index=False)

In [3]:
import pandas as pd
import os
import shutil
import json

# Load the CSV file with subject-to-group mapping
csv_path = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\zaynul23_3_6_2026_2_32_57.csv'
df = pd.read_csv(csv_path)

# Load the group labels
grp_json_path = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\grpdat.json'
with open(grp_json_path, 'r') as f:
    grp_labels = json.load(f)

# Create a dictionary mapping subject IDs to their groups
subject_to_group = dict(zip(df['Subject'], df['Group']))

# Path to the NUDataSharing directory
nu_data_path = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NUDataSharing'

# Groups to delete entirely
delete_groups = [2, 4]

# Track statistics
deleted_folders = 0
kept_folders = 0
modified_folders = 0
folders_with_only_mpr1 = 0

# Iterate through all subject folders
for subject_folder in os.listdir(nu_data_path):
    subject_path = os.path.join(nu_data_path, subject_folder)
    
    # Skip if not a directory
    if not os.path.isdir(subject_path):
        continue
    
    # Get the group for this subject
    if subject_folder not in subject_to_group:
        print(f"Warning: Subject {subject_folder} not found in CSV")
        continue
    
    group = subject_to_group[subject_folder]
    
    # If subject is in group 2 or 4, delete the entire folder
    if group in delete_groups:
        try:
            shutil.rmtree(subject_path)
            print(f"Deleted entire folder: {subject_folder} (Group {group}: {grp_labels.get(str(group), 'Unknown')})")
            deleted_folders += 1
        except Exception as e:
            print(f"Error deleting {subject_folder}: {e}")
    else:
        # For other groups, keep only MPR1
        # First, find the session subfolder(s)
        session_folders = [d for d in os.listdir(subject_path) if os.path.isdir(os.path.join(subject_path, d))]
        
        for session_folder in session_folders:
            session_path = os.path.join(subject_path, session_folder)
            
            # List the scan types in this session
            scan_types = [d for d in os.listdir(session_path) if os.path.isdir(os.path.join(session_path, d))]
            
            mpr1_exists = 'MPR1' in scan_types
            
            # Delete all scan types except MPR1
            for scan_type in scan_types:
                if scan_type != 'MPR1':
                    scan_path = os.path.join(session_path, scan_type)
                    try:
                        shutil.rmtree(scan_path)
                        print(f"Deleted {scan_type} from {subject_folder} (Group {group}: {grp_labels.get(str(group), 'Unknown')})")
                        modified_folders += 1
                    except Exception as e:
                        print(f"Error deleting {scan_type} from {subject_folder}: {e}")
            
            if mpr1_exists:
                folders_with_only_mpr1 += 1
        
        kept_folders += 1

print("\n" + "="*60)
print("CLEANUP SUMMARY")
print("="*60)
print(f"Total folders deleted (Groups 2 & 4): {deleted_folders}")
print(f"Total folders kept (Groups 1, 3, 5): {kept_folders}")
print(f"Total scans modified (non-MPR1 deleted): {modified_folders}")
print(f"Folders with MPR1 preserved: {folders_with_only_mpr1}")
print("="*60)

Deleted entire folder: CC0005 (Group 2.0: Unknown)
Deleted entire folder: CC0196 (Group 2.0: Unknown)
Deleted FLASH1 from CC0197 (Group 1.0: Unknown)
Deleted MPR2 from CC0197 (Group 1.0: Unknown)
Deleted MPR2 from CC0202 (Group 1.0: Unknown)
Deleted MPR3 from CC0202 (Group 1.0: Unknown)
Deleted MPR4 from CC0202 (Group 1.0: Unknown)
Deleted entire folder: CC0212 (Group 2.0: Unknown)
Deleted FLASH1 from CC0239 (Group 3.0: Unknown)
Deleted MPR2 from CC0239 (Group 3.0: Unknown)
Deleted MPR3 from CC0239 (Group 3.0: Unknown)
Deleted MPR4 from CC0239 (Group 3.0: Unknown)
Deleted FLASH1 from CC0311 (Group 5.0: Unknown)
Deleted MPR2 from CC0311 (Group 5.0: Unknown)
Deleted MPR3 from CC0311 (Group 5.0: Unknown)
Deleted MPR4 from CC0311 (Group 5.0: Unknown)
Deleted FLASH1 from CC0325 (Group 3.0: Unknown)
Deleted MPR2 from CC0325 (Group 3.0: Unknown)
Deleted MPR3 from CC0325 (Group 3.0: Unknown)
Deleted MPR4 from CC0325 (Group 3.0: Unknown)
Deleted FLASH1 from CC0385 (Group 3.0: Unknown)
Deleted M

In [ ]:
import os
import shutil
import pandas as pd

# Define paths
analyse_path = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\analyse'
csv_path = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\zaynul23_3_6_2026_2_32_57.csv'
control_siblings_path = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\control_siblings'

# Read the CSV file
df = pd.read_csv(csv_path)

# Create a mapping of subject to group
subject_to_group = dict(zip(df['Subject'], df['Group']))

print(f"Total subjects in CSV: {len(subject_to_group)}\n")

# Create destination directory if needed
os.makedirs(control_siblings_path, exist_ok=True)

renamed_count = 0
moved_count = 0
skipped_count = 0

# Iterate through all folders in analyse directory
for folder_name in os.listdir(analyse_path):
    folder_path = os.path.join(analyse_path, folder_name)
    
    # Skip if not a directory
    if not os.path.isdir(folder_path):
        continue
    
    # Check if this folder is in the CSV
    if folder_name not in subject_to_group:
        print(f"⚠️  Not in CSV: {folder_name}")
        skipped_count += 1
        continue
    
    group = subject_to_group[folder_name]
    group_int = int(group)  # Convert float to int
    
    # Create new name with group number
    new_name = f"{folder_name}_{group_int}"
    new_path = os.path.join(analyse_path, new_name)
    
    # Rename the folder (in place, in analyse folder)
    try:
        os.rename(folder_path, new_path)
        print(f"✓ Renamed: {folder_name} → {new_name}")
        renamed_count += 1
        
        # If it's group 5, also move to control_siblings
        if group_int == 5:
            dest_path = os.path.join(control_siblings_path, new_name)
            shutil.move(new_path, dest_path)
            print(f"  ↳ Moved to control_siblings")
            moved_count += 1
    
    except Exception as e:
        print(f"❌ Error processing {folder_name}: {str(e)}")

print(f"\n{'='*60}")
print(f"Summary:")
print(f"Folders renamed: {renamed_count}")
print(f"Group 5 folders moved to control_siblings: {moved_count}")
print(f"Folders not in CSV: {skipped_count}")
print(f"{'='*60}")

Total subjects in CSV: 463

✓ Renamed: CC0197 → CC0197_1.0
✓ Renamed: CC0202 → CC0202_1.0
✓ Renamed: CC0239 → CC0239_3.0
✓ Renamed: CC0311 → CC0311_5.0
  ↳ Moved to control_siblings
✓ Renamed: CC0325 → CC0325_3.0
✓ Renamed: CC0385 → CC0385_3.0
✓ Renamed: CC0432 → CC0432_1.0
✓ Renamed: CC0475 → CC0475_3.0
✓ Renamed: CC0486 → CC0486_5.0
  ↳ Moved to control_siblings
✓ Renamed: CC0600 → CC0600_3.0
✓ Renamed: CC0683 → CC0683_1.0
✓ Renamed: CC0717 → CC0717_3.0
✓ Renamed: CC0857 → CC0857_5.0
  ↳ Moved to control_siblings
✓ Renamed: CC0873 → CC0873_5.0
  ↳ Moved to control_siblings
✓ Renamed: CC0949 → CC0949_5.0
  ↳ Moved to control_siblings
✓ Renamed: CC1085 → CC1085_3.0
✓ Renamed: CC1197 → CC1197_3.0
✓ Renamed: CC1381 → CC1381_1.0
✓ Renamed: CC1386 → CC1386_1.0
✓ Renamed: CC1496 → CC1496_3.0
✓ Renamed: CC1549 → CC1549_1.0
✓ Renamed: CC1570 → CC1570_3.0
✓ Renamed: CC1709 → CC1709_3.0
✓ Renamed: CC1960 → CC1960_3.0
✓ Renamed: CC2038 → CC2038_5.0
  ↳ Moved to control_siblings
✓ Renamed: CC2191

In [4]:
import os

analyse_path = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\analyse'

grp1_count = 0
grp3_count = 0

# Count folders with _1 and _3 suffixes
for folder_name in os.listdir(analyse_path):
    folder_path = os.path.join(analyse_path, folder_name)
    if os.path.isdir(folder_path):
        if folder_name.endswith('_1'):
            grp1_count += 1
        elif folder_name.endswith('_3'):
            grp3_count += 1

print(f"Group 1 folders: {grp1_count}")
print(f"Group 3 folders: {grp3_count}")
print(f"Total (Group 1 + 3): {grp1_count + grp3_count}")

Group 1 folders: 147
Group 3 folders: 137
Total (Group 1 + 3): 284


In [ ]:
#convert analyse to nifti
import os
import nibabel as nib

# input root folder (analyse dataset)
input_root = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\analyse"

# output folder
output_root = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI"
os.makedirs(output_root, exist_ok=True)

# iterate through each subject folder
for subject in os.listdir(input_root):

    subject_path = os.path.join(input_root, subject)

    if not os.path.isdir(subject_path):
        continue

    hdr_file = None

    # find hdr file inside subject folder
    for file in os.listdir(subject_path):
        if file.endswith(".hdr"):
            hdr_file = os.path.join(subject_path, file)
            break

    if hdr_file is None:
        print(f"No HDR found in {subject}")
        continue

    try:
        # load analyze image (reads hdr + img automatically)
        img = nib.load(hdr_file)

        # output filename
        output_path = os.path.join(output_root, f"{subject}.nii.gz")

        # save as nifti
        nib.save(img, output_path)

        print(f"Converted {subject}")

    except Exception as e:
        print(f"Error processing {subject}: {e}")

print("Conversion complete.")

Converted CC0197_1
Converted CC0202_1
Converted CC0239_3
Converted CC0325_3
Converted CC0385_3
Converted CC0432_1
Converted CC0475_3
Converted CC0600_3
Converted CC0683_1
Converted CC0717_3
Converted CC1085_3
Converted CC1197_3
Converted CC1381_1
Converted CC1386_1
Converted CC1496_3
Converted CC1549_1
Converted CC1570_3
Converted CC1709_3
Converted CC1960_3
Converted CC2191_1
Converted CC2199_1
Converted CC2263_1
Converted CC2430_1
Converted CC2442_3
Converted CC2485_1
Converted CC2606_3
Converted CC2825_3
Converted CC3037_3
Converted CC3493_1
Converted CC3499_1
Converted CC3915_3
Converted CC3945_3
Converted CC4094_3
Converted CC4183_3
Converted CC4189_3
Converted CC4431_3
Converted CC4526_3
Converted CC4896_3
Converted CC5006_3
Converted CC5041_1
Converted CC5111_3
Converted CC5479_1
Converted CC5529_3
Converted CC5562_3
Converted CC5721_1
Converted CC5884_3
Converted CC5892_3
Converted CC5989_3
Converted CC6167_1
Converted CC6182_1
Converted CC6183_3
Converted CC6238_1
Converted CC

<module 'ntpath' from 'C:\\Users\\zaynu\\AppData\\Local\\Programs\\Python\\Python310\\lib\\ntpath.py'>

In [8]:
import os

# Path to NIFTI directory
nifti_path = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI'

# Count all files and those with suffix 3 or 1
total_files = 0
suffix_3_count = 0
suffix_1_count = 0

# Walk through all subdirectories
for root, dirs, files in os.walk(nifti_path):
    for file in files:
        total_files += 1
        
        # Remove all extensions to get the base name
        # For CC0197_1.nii.gz, this gives us CC0197_1
        base_name = file
        while '.' in base_name:
            base_name = os.path.splitext(base_name)[0]
        
        # Check if base name ends with _3 or _1
        if base_name.endswith('_3_0'):
            suffix_3_count += 1
        elif base_name.endswith('_1_0'):
            suffix_1_count += 1

print("="*60)
print("NIFTI FOLDER FILE COUNT SUMMARY")
print("="*60)
print(f"Total files in NIFTI folder: {total_files}")
print(f"Files with suffix '_3_0': {suffix_3_count}")
print(f"Files with suffix '_1_0': {suffix_1_count}")
print(f"Files with other suffixes: {total_files - suffix_3_count - suffix_1_count}")
print("="*60)

NIFTI FOLDER FILE COUNT SUMMARY
Total files in NIFTI folder: 163
Files with suffix '_3_0': 67
Files with suffix '_1_0': 96
Files with other suffixes: 0


In [8]:
import os

# Define path
nifti_dir = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_26\scans\NIFTI'

# Group files by suffix
files_1_1 = []
files_3_1 = []
other_files = []

# Read all files in the directory
for file in os.listdir(nifti_dir):
    file_path = os.path.join(nifti_dir, file)
    
    # Skip if not a file
    if not os.path.isfile(file_path):
        continue
    
    # Remove .nii.gz extension
    if file.endswith('.nii.gz'):
        base_name = file[:-7]
    else:
        base_name = file
    
    # Group by suffix
    if base_name.endswith('_1_1'):
        files_1_1.append(file)
    elif base_name.endswith('_3_1'):
        files_3_1.append(file)
    else:
        other_files.append(file)

print("="*60)
print("FILE GROUPING SUMMARY")
print("="*60)
print(f"\nFiles ending with _1_1: {len(files_1_1)}")
for f in sorted(files_1_1):
    print(f"  • {f}")

print(f"\nFiles ending with _3_1: {len(files_3_1)}")
for f in sorted(files_3_1):
    print(f"  • {f}")

print(f"\nOther files: {len(other_files)}")
for f in sorted(other_files):
    print(f"  • {f}")

print(f"\n{'='*60}")
print(f"Total files: {len(files_1_1) + len(files_3_1) + len(other_files)}")
print(f"{'='*60}")

FILE GROUPING SUMMARY

Files ending with _1_1: 26
  • sub-2218A_ses-0001_T1w_1_1.nii.gz
  • sub-2227A_ses-0001_T1w_1_1.nii.gz
  • sub-2245A_ses-0001_T1w_1_1.nii.gz
  • sub-2246A_ses-0001_T1w_1_1.nii.gz
  • sub-2252A_ses-0001_T1w_1_1.nii.gz
  • sub-2266A_ses-0001_T1w_1_1.nii.gz
  • sub-2295A_ses-0001_T1w_1_1.nii.gz
  • sub-2318A_ses-0001_T1w_1_1.nii.gz
  • sub-2326A_ses-0001_T1w_1_1.nii.gz
  • sub-2336A_ses-0001_T1w_1_1.nii.gz
  • sub-2343A_ses-0001_T1w_1_1.nii.gz
  • sub-2351A_ses-0001_T1w_1_1.nii.gz
  • sub-2365A_ses-0001_T1w_1_1.nii.gz
  • sub-2367A_ses-0001_T1w_1_1.nii.gz
  • sub-2379A_ses-0001_T1w_1_1.nii.gz
  • sub-2397A_ses-0001_T1w_1_1.nii.gz
  • sub-2418A_ses-0001_T1w_1_1.nii.gz
  • sub-2419A_ses-0001_T1w_1_1.nii.gz
  • sub-2431A_ses-0001_T1w_1_1.nii.gz
  • sub-2448A_ses-0001_T1w_1_1.nii.gz
  • sub-2476A_ses-0001_T1w_1_1.nii.gz
  • sub-2498A_ses-0001_T1w_1_1.nii.gz
  • sub-2505A_ses-0001_T1w_1_1.nii.gz
  • sub-2514A_ses-0001_T1w_1_1.nii.gz
  • sub-2578A_ses-0001_T1w_1_1.nii.gz


In [1]:
"""
MRI Reorientation Augmenter
============================
Takes a single .nii / .nii.gz scan, applies a slight reorientation,
and saves the augmented version to an output directory.

Augmentations (mild, realistic dataset variation):
  - Small rotation on each axis (default ±5 degrees)
  - Optional axis flip  (simulates LR/AP/IS convention differences)
  - Optional axis permutation (simulates different slice orientation)

Each output file is saved as:  <original_name>_aug.nii.gz

Requirements:
    pip install nibabel numpy scipy
"""

import os
import sys
import numpy as np
import nibabel as nib
from scipy.ndimage import rotate
import warnings
warnings.filterwarnings("ignore")


# ═══════════════════════════════════════════════════════════════════════════════
#  CONFIG — edit this block only
# ═══════════════════════════════════════════════════════════════════════════════

SCAN_PATH        = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI\CC0202_1.nii.gz"  # path to a single .nii / .nii.gz file
OUTPUT_DIR       = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\test"        # augmented file saved here

ROTATION_MAX_DEG = 5      # max rotation per axis in degrees (keep small: 3–10)
APPLY_AXIS_FLIP  = True   # flip one axis (simulates orientation convention change)
APPLY_AXIS_PERM  = False  # permute axes (stronger augmentation, stress-test only)
RANDOM_SEED      = 42     # fixed seed = reproducible augmentations

# ═══════════════════════════════════════════════════════════════════════════════


def find_scans(directory: str) -> list:
    """Recursively find all .nii and .nii.gz files in a directory."""
    found = []
    for root, _, files in os.walk(directory):
        for f in sorted(files):
            if f.endswith(".nii.gz") or f.endswith(".nii"):
                found.append(os.path.join(root, f))
    return found


def load_volume(path: str):
    """Load NIfTI, return (3D float32 array, original nibabel image)."""
    img  = nib.load(path)
    data = np.asarray(img.dataobj, dtype=np.float32)
    # Squeeze trailing size-1 dimensions e.g. (128,256,256,1) -> (128,256,256)
    while data.ndim > 3 and data.shape[-1] == 1:
        data = data[..., 0]
    if data.ndim != 3:
        raise ValueError(f"Expected 3D volume after squeezing, got {data.shape}")
    return data, img


def augment(vol: np.ndarray, seed: int) -> tuple:
    """
    Apply mild reorientation augmentations to a 3D volume.
    Each scan gets a unique seed derived from the global seed + its index,
    so every scan gets a slightly different augmentation.

    Returns:
        augmented numpy array (float32)
        list of strings describing what was applied
    """
    rng = np.random.default_rng(seed)
    out = vol.copy()
    log = []

    # Small rotation on each plane
    for (ax0, ax1), name in [((0,1), "XY (axial)"),
                              ((0,2), "XZ (coronal)"),
                              ((1,2), "YZ (sagittal)")]:
        angle = rng.uniform(-ROTATION_MAX_DEG, ROTATION_MAX_DEG)
        out   = rotate(out, angle=angle, axes=(ax0, ax1),
                       reshape=False, order=1, mode="constant", cval=0.0)
        log.append(f"  Rotation {name:15s}: {angle:+.2f}°")

    # Optional axis flip
    if APPLY_AXIS_FLIP:
        axis = int(rng.integers(0, 3))
        out  = np.flip(out, axis=axis).copy()
        log.append(f"  Axis flip              : axis {axis}")

    # Optional axis permutation
    if APPLY_AXIS_PERM:
        perm = rng.permutation(3).tolist()
        out  = np.transpose(out, perm)
        log.append(f"  Axis permutation       : {perm}")

    return out.astype(np.float32), log


def save_volume(data: np.ndarray, ref_img, out_path: str) -> None:
    """Save a numpy array as NIfTI, preserving the original header and affine."""
    new_img = nib.Nifti1Image(data, affine=ref_img.affine, header=ref_img.header)
    new_img.header.set_data_shape(data.shape)
    nib.save(new_img, out_path)


def output_path(input_path: str, input_dir: str, output_dir: str) -> str:
    """
    Mirror the input directory structure in the output directory.
    e.g. input_dir/subdir/scan.nii.gz -> output_dir/subdir/scan_aug.nii.gz
    """
    rel = os.path.relpath(input_path, input_dir)
    base = rel
    if base.endswith(".nii.gz"):
        base = base[:-7] + "_aug.nii.gz"
    elif base.endswith(".nii"):
        base = base[:-4] + "_aug.nii.gz"
    return os.path.join(output_dir, base)


def main():
    if not os.path.isfile(SCAN_PATH):
        print(f"ERROR: Scan not found: {SCAN_PATH}")
        sys.exit(1)
    if not (SCAN_PATH.endswith(".nii.gz") or SCAN_PATH.endswith(".nii")):
        print(f"ERROR: File must be .nii or .nii.gz: {SCAN_PATH}")
        sys.exit(1)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    base = os.path.basename(SCAN_PATH)
    if base.endswith(".nii.gz"):
        out_name = base[:-7] + "_aug.nii.gz"
    else:
        out_name = base[:-4] + "_aug.nii.gz"
    out_path = os.path.join(OUTPUT_DIR, out_name)

    print(f"\n{'='*60}")
    print(f"  MRI REORIENTATION AUGMENTER")
    print(f"{'='*60}")
    print(f"  Input  : {SCAN_PATH}")
    print(f"  Output : {out_path}")
    print(f"  Max rotation : ±{ROTATION_MAX_DEG}°  |  "
          f"Flip: {APPLY_AXIS_FLIP}  |  Permute: {APPLY_AXIS_PERM}")
    print(f"{'='*60}\n")

    print(f"  Loading...")
    try:
        vol, ref_img = load_volume(SCAN_PATH)
    except Exception as e:
        print(f"  ERROR loading scan: {e}")
        sys.exit(1)
    print(f"  Shape: {vol.shape}")

    print(f"  Augmenting...")
    aug_vol, aug_log = augment(vol, seed=RANDOM_SEED)
    for line in aug_log:
        print(line)

    print(f"\n  Saving...")
    try:
        save_volume(aug_vol, ref_img, out_path)
        print(f"  Saved → {out_path}")
    except Exception as e:
        print(f"  SAVE ERROR: {e}")
        sys.exit(1)

    print(f"\n{'='*60}")
    print(f"  Done.")
    print(f"{'='*60}\n")


if __name__ == "__main__":
    main()



  MRI REORIENTATION AUGMENTER
  Input  : D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI\CC0202_1.nii.gz
  Output : D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\test\CC0202_1_aug.nii.gz
  Max rotation : ±5°  |  Flip: True  |  Permute: False

  Loading...
  Shape: (128, 256, 256)
  Augmenting...
  Rotation XY (axial)     : +2.74°
  Rotation XZ (coronal)   : -0.61°
  Rotation YZ (sagittal)  : +3.59°
  Axis flip              : axis 0

  Saving...
  Saved → D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\test\CC0202_1_aug.nii.gz

  Done.



In [10]:
import os
len(os.listdir(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_1_71\scans"))

71

In [11]:
"""
MRI Scan Organiser
===================
Reads the nested NIFTI folder structure:
  NIFTI/sub-XXXX*/ses-XXXX/anat/*.nii.gz

For each scan:
  - Looks up the subject's group from participants.csv
  - Renames the file:  <original_stem>_<group_number>_1.nii.gz
  - FESZ  (group 1) → moved into NIFTI/
  - HC    (group 3) → moved into NIFTI/
  - FEAFF (group 6) → moved into other_nifti/
  - Unknown subject  → skipped (file left, reported)

After all scans are moved:
  - All subfolders inside NIFTI/ are deleted (including any leftover files)

CONFIG — edit the paths below before running.
"""

import os
import re
import csv
import shutil

# ═══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════════════════════

NIFTI_DIR        = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_26\scans\NIFTI"
OTHER_NIFTI_DIR  = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_26\scans\other_nifti"
PARTICIPANTS_CSV = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_26\participants.csv"

GROUP_MAP = {
    "FESZ":  1,
    "FEAFF": 6,
    "HC":    3,
}

# ═══════════════════════════════════════════════════════════════════════════════


def load_subject_groups(csv_path: str) -> dict:
    """
    Returns dict mapping RECID string → group number (int).
    e.g. {"2218": 1, "2242": 3, "2237": 6, ...}
    """
    mapping = {}
    with open(csv_path, encoding="utf-8-sig") as f:
        reader = csv.DictReader(f)
        for row in reader:
            recid   = row["RECID"].strip()
            subtype = row["SUBTYPE2"].strip().upper().replace(" ", "")
            grp_num = GROUP_MAP.get(subtype)
            if grp_num is not None:
                mapping[recid] = grp_num
    return mapping


def find_nii_files(root: str) -> list:
    """
    Recursively find only T1w .nii/.nii.gz files inside subfolders.
    Ignores anything already sitting in root itself (already processed).
    Ignores all other scan types (bold, dwi, fmap, etc.) and non-NIfTI files.
    Those will be wiped when the subfolders are deleted.
    """
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        if os.path.abspath(dirpath) == os.path.abspath(root):
            continue
        for fname in filenames:
            is_nifti = fname.endswith(".nii.gz") or fname.endswith(".nii")
            is_t1w   = "T1w" in fname
            if is_nifti and is_t1w:
                found.append(os.path.join(dirpath, fname))
    return found


def extract_recid(folder_path: str) -> str | None:
    """
    Walk up the path components looking for a sub-XXXX* folder name
    and extract the numeric RECID from it.
    e.g. .../sub-2218A/... → "2218"
    """
    parts = folder_path.replace("\\", "/").split("/")
    for part in parts:
        m = re.match(r"sub-(\d+)", part)
        if m:
            return m.group(1)
    return None


def build_output_name(original_fname: str, grp_num: int) -> str:
    """
    original_fname: e.g. sub-2218A_ses-0001_T1w.nii.gz
    returns:             sub-2218A_ses-0001_T1w_1_1.nii.gz
    """
    if original_fname.endswith(".nii.gz"):
        stem = original_fname[:-7]
        ext  = ".nii.gz"
    else:
        stem = original_fname[:-4]
        ext  = ".nii"
    return f"{stem}_{grp_num}_1{ext}"


def safe_dest(dest_path: str) -> str:
    """If dest_path already exists add a numeric suffix to avoid overwrite."""
    if not os.path.exists(dest_path):
        return dest_path
    base, ext = dest_path, ""
    if dest_path.endswith(".nii.gz"):
        base, ext = dest_path[:-7], ".nii.gz"
    elif dest_path.endswith(".nii"):
        base, ext = dest_path[:-4], ".nii"
    i = 2
    while os.path.exists(f"{base}_{i}{ext}"):
        i += 1
    return f"{base}_{i}{ext}"


def main():
    print("\n" + "="*60)
    print("  MRI SCAN ORGANISER")
    print("="*60)

    # Validate paths
    if not os.path.isdir(NIFTI_DIR):
        print(f"ERROR: NIFTI_DIR not found:\n  {NIFTI_DIR}")
        return
    if not os.path.isfile(PARTICIPANTS_CSV):
        print(f"ERROR: participants.csv not found:\n  {PARTICIPANTS_CSV}")
        return

    # Load group mapping
    subject_groups = load_subject_groups(PARTICIPANTS_CSV)
    print(f"\n  Loaded {len(subject_groups)} subjects from participants.csv")
    print(f"  FESZ (1): {sum(1 for v in subject_groups.values() if v==1)}  "
          f"HC (3): {sum(1 for v in subject_groups.values() if v==3)}  "
          f"FEAFF (6): {sum(1 for v in subject_groups.values() if v==6)}")

    # Create output dirs
    os.makedirs(NIFTI_DIR, exist_ok=True)
    os.makedirs(OTHER_NIFTI_DIR, exist_ok=True)

    # Find all scans inside subfolders
    scan_files = find_nii_files(NIFTI_DIR)
    print(f"\n  Found {len(scan_files)} scan file(s) in subfolders\n")

    moved      = []
    skipped    = []

    for src_path in scan_files:
        fname  = os.path.basename(src_path)
        recid  = extract_recid(src_path)

        if recid is None:
            print(f"  [SKIP] Cannot extract subject ID from path:\n    {src_path}")
            skipped.append(src_path)
            continue

        grp_num = subject_groups.get(recid)
        if grp_num is None:
            print(f"  [SKIP] Subject {recid} not found in participants.csv:\n    {fname}")
            skipped.append(src_path)
            continue

        out_name = build_output_name(fname, grp_num)

        if grp_num == 6:   # FEAFF → other_nifti
            dest_dir = OTHER_NIFTI_DIR
            tag = "FEAFF→other_nifti"
        else:              # FESZ or HC → NIFTI root
            dest_dir = NIFTI_DIR
            tag = f"{'FESZ' if grp_num==1 else 'HC'}→NIFTI"

        dest_path = safe_dest(os.path.join(dest_dir, out_name))
        shutil.move(src_path, dest_path)
        moved.append((src_path, dest_path))
        print(f"  [{tag}]  {fname}")
        print(f"          → {os.path.basename(dest_path)}")

    # ── Delete all subfolders (and any remaining files in them) ──────────────
    print(f"\n  Cleaning up subfolders...")
    deleted_dirs = 0
    for entry in os.listdir(NIFTI_DIR):
        full = os.path.join(NIFTI_DIR, entry)
        if os.path.isdir(full):
            shutil.rmtree(full)
            print(f"  [DELETED] {entry}/")
            deleted_dirs += 1

    # ── Summary ──────────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  DONE")
    print(f"{'='*60}")
    print(f"  Scans moved   : {len(moved)}")
    print(f"  Scans skipped : {len(skipped)}")
    print(f"  Folders deleted : {deleted_dirs}")
    print(f"\n  Resultant structure:")
    print(f"  NIFTI/")
    for _, dst in moved:
        if os.path.abspath(os.path.dirname(dst)) == os.path.abspath(NIFTI_DIR):
            print(f"    {os.path.basename(dst)}")
    print(f"  other_nifti/")
    for _, dst in moved:
        if os.path.abspath(os.path.dirname(dst)) == os.path.abspath(OTHER_NIFTI_DIR):
            print(f"    {os.path.basename(dst)}")
    print()

    if skipped:
        print(f"  Skipped files (no group match):")
        for s in skipped:
            print(f"    {s}")
    print()


if __name__ == "__main__":
    main()



  MRI SCAN ORGANISER

  Loaded 60 subjects from participants.csv
  FESZ (1): 26  HC (3): 26  FEAFF (6): 8

  Found 38 scan file(s) in subfolders

  [FESZ→NIFTI]  sub-2218A_ses-0001_T1w.nii.gz
          → sub-2218A_ses-0001_T1w_1_1.nii.gz
  [HC→NIFTI]  sub-2221A_ses-0001_T1w.nii.gz
          → sub-2221A_ses-0001_T1w_3_1.nii.gz
  [FESZ→NIFTI]  sub-2227A_ses-0001_T1w.nii.gz
          → sub-2227A_ses-0001_T1w_1_1.nii.gz
  [FEAFF→other_nifti]  sub-2237A_ses-0001_T1w.nii.gz
          → sub-2237A_ses-0001_T1w_6_1.nii.gz
  [HC→NIFTI]  sub-2242A_ses-0001_T1w.nii.gz
          → sub-2242A_ses-0001_T1w_3_1.nii.gz
  [FESZ→NIFTI]  sub-2245A_ses-0001_T1w.nii.gz
          → sub-2245A_ses-0001_T1w_1_1.nii.gz
  [FESZ→NIFTI]  sub-2246A_ses-0001_T1w.nii.gz
          → sub-2246A_ses-0001_T1w_1_1.nii.gz
  [HC→NIFTI]  sub-2247A_ses-0001_T1w.nii.gz
          → sub-2247A_ses-0001_T1w_3_1.nii.gz
  [FESZ→NIFTI]  sub-2252A_ses-0001_T1w.nii.gz
          → sub-2252A_ses-0001_T1w_1_1.nii.gz
  [HC→NIFTI]  sub-2253A_

In [12]:
import os

# Define path
nifti_dir = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_26\scans\NIFTI'

# Group files by suffix
files_1_1 = []
files_3_1 = []
other_files = []

# Read all files in the directory
for file in os.listdir(nifti_dir):
    file_path = os.path.join(nifti_dir, file)
    
    # Skip if not a file
    if not os.path.isfile(file_path):
        continue
    
    # Remove .nii.gz extension
    if file.endswith('.nii.gz'):
        base_name = file[:-7]
    else:
        base_name = file
    
    # Group by suffix
    if base_name.endswith('_1_1'):
        files_1_1.append(file)
    elif base_name.endswith('_3_1'):
        files_3_1.append(file)
    else:
        other_files.append(file)

print("="*60)
print("FILE GROUPING SUMMARY")
print("="*60)
print(f"\nFiles ending with _1_1: {len(files_1_1)}")
for f in sorted(files_1_1):
    print(f"  • {f}")

print(f"\nFiles ending with _3_1: {len(files_3_1)}")
for f in sorted(files_3_1):
    print(f"  • {f}")

print(f"\nOther files: {len(other_files)}")
for f in sorted(other_files):
    print(f"  • {f}")

print(f"\n{'='*60}")
print(f"Total files: {len(files_1_1) + len(files_3_1) + len(other_files)}")
print(f"{'='*60}")

FILE GROUPING SUMMARY

Files ending with _1_1: 16
  • sub-2218A_ses-0001_T1w_1_1.nii.gz
  • sub-2227A_ses-0001_T1w_1_1.nii.gz
  • sub-2245A_ses-0001_T1w_1_1.nii.gz
  • sub-2246A_ses-0001_T1w_1_1.nii.gz
  • sub-2252A_ses-0001_T1w_1_1.nii.gz
  • sub-2266A_ses-0001_T1w_1_1.nii.gz
  • sub-2295A_ses-0001_T1w_1_1.nii.gz
  • sub-2318A_ses-0001_T1w_1_1.nii.gz
  • sub-2326A_ses-0001_T1w_1_1.nii.gz
  • sub-2336A_ses-0001_T1w_1_1.nii.gz
  • sub-2343A_ses-0001_T1w_1_1.nii.gz
  • sub-2351A_ses-0001_T1w_1_1.nii.gz
  • sub-2365A_ses-0001_T1w_1_1.nii.gz
  • sub-2367A_ses-0001_T1w_1_1.nii.gz
  • sub-2379A_ses-0001_T1w_1_1.nii.gz
  • sub-2397A_ses-0001_T1w_1_1.nii.gz

Files ending with _3_1: 18
  • sub-2221A_ses-0001_T1w_3_1.nii.gz
  • sub-2242A_ses-0001_T1w_3_1.nii.gz
  • sub-2247A_ses-0001_T1w_3_1.nii.gz
  • sub-2253A_ses-0001_T1w_3_1.nii.gz
  • sub-2259A_ses-0001_T1w_3_1.nii.gz
  • sub-2262A_ses-0001_T1w_3_1.nii.gz
  • sub-2268A_ses-0001_T1w_3_1.nii.gz
  • sub-2269A_ses-0001_T1w_3_1.nii.gz
  • sub-22

In [6]:
import os

# The numbers to check
numbers = [
2221,2242,2247,2253,2259,2262,2268,2269,2286,2320,2328,2342,2350,2355,2372,2384,2387,2406,2426,2430,2479,2494,2496,2512,2538,2574,
]
numbers = [
2218,
2295,
2336,
2498,
2505,
2578,
2418,
2245,
2246,
2252,
2365,
2367,
2431,
2514,
2581,
2227,
2318,
2326,
2343,
2351,
2379,
2448,
2476,
2266,
2397,
2419,
]

# Directory to check
nifti_dir = r'D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_26\scans\NIFTI'

# Get all filenames in the directory
all_files = os.listdir(nifti_dir)

# Check which numbers are missing
missing_numbers = []
for num in numbers:
    num_str = str(num)
    found = any(num_str in filename for filename in all_files)
    if not found:
        missing_numbers.append(num)

print("Numbers NOT found in any filename:")
print(missing_numbers)
print(f"\nTotal missing: {len(missing_numbers)} out of {len(numbers)}")

Numbers NOT found in any filename:
[2581]

Total missing: 1 out of 26


In [2]:
import os
import shutil
import csv

# ── PATHS ──────────────────────────────────────────────────────────────────────
NIFTI_DIR     = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_16\scans\NIFTI"
OTHER_DIR     = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_16\scans\other_nifti"
TSV_PATH      = os.path.join(NIFTI_DIR, "participants.tsv")

# ── GROUP MAPPING ──────────────────────────────────────────────────────────────
# TSV groupID 1 = Bipolar      → label 4  → goes to OTHER_DIR
# TSV groupID 2 = Schizophrenia → label 1  → stays in NIFTI_DIR
GROUP_LABEL_MAP = {
    "1": "4",   # Bipolar
    "2": "1",   # Schizophrenia
}

# ── SETUP ──────────────────────────────────────────────────────────────────────
os.makedirs(OTHER_DIR, exist_ok=True)

# ── READ TSV ───────────────────────────────────────────────────────────────────
subject_groups = {}   # { "sub-B04": "1", "sub-S06": "2", ... }

with open(TSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        subj_id  = row["participant_id"].strip()
        group_id = row["groupID"].strip()
        subject_groups[subj_id] = group_id

print(f"Loaded {len(subject_groups)} subjects from TSV.\n")

# ── PROCESS EACH SUBJECT ───────────────────────────────────────────────────────
moved      = []
skipped    = []
not_found  = []

for subj_id, tsv_group in subject_groups.items():

    # Build path to rec-1 T1w file
    src_file = os.path.join(
        NIFTI_DIR, subj_id, "anat",
        f"{subj_id}_rec-1_T1w.nii.gz"
    )

    if not os.path.isfile(src_file):
        print(f"  [SKIP] {subj_id}: rec-1_T1w.nii.gz not found at expected path.")
        not_found.append(subj_id)
        continue

    # Determine label and destination
    label        = GROUP_LABEL_MAP.get(tsv_group)
    if label is None:
        print(f"  [SKIP] {subj_id}: unknown groupID '{tsv_group}' in TSV.")
        skipped.append(subj_id)
        continue

    new_filename = f"{subj_id}_rec-1_T1w_{label}_2.nii.gz"

    if tsv_group == "1":          # Bipolar → other_nifti
        dest_file = os.path.join(OTHER_DIR, new_filename)
    else:                         # Schizophrenia → stay in NIFTI
        dest_file = os.path.join(NIFTI_DIR, new_filename)

    # Copy file to destination with new name
    shutil.copy2(src_file, dest_file)
    print(f"  [OK] {subj_id} (group {tsv_group} → label {label})")
    print(f"       {src_file}")
    print(f"    -> {dest_file}")

    # Delete entire subject folder
    subj_folder = os.path.join(NIFTI_DIR, subj_id)
    shutil.rmtree(subj_folder)
    print(f"       Deleted folder: {subj_folder}\n")

    moved.append(subj_id)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("=" * 60)
print(f"Done.")
print(f"  Successfully processed : {len(moved)} subjects")
print(f"  Skipped (no file)      : {len(not_found)} subjects -> {not_found}")
print(f"  Skipped (unknown group): {len(skipped)} subjects -> {skipped}")
print(f"\nSchizophrenia files (label _1_2) -> {NIFTI_DIR}")
print(f"Bipolar files       (label _4_2) -> {OTHER_DIR}")

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\zaynu\\Documents\\coding\\PS\\2-2\\schizobrain-Scan\\MRI_data\\open_neuro\\open_neuro_3_58\\scans\\NIFTI\\participants.tsv'

In [4]:
import os
import shutil
import csv

# ── PATHS ──────────────────────────────────────────────────────────────────────
NIFTI_DIR  = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI"
OTHER_DIR  = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\other_nifti"
TSV_PATH   = os.path.join(NIFTI_DIR, "participants.tsv")

# ── GROUP MAPPING ──────────────────────────────────────────────────────────────
# condit value  │ action              │ label │ destination
# ─────────────────────────────────────────────────────────
# SCZ           │ extract + rename    │  1    │ NIFTI_DIR
# CON           │ extract + rename    │  3    │ NIFTI_DIR
# CON-SIB       │ extract + rename    │  6    │ OTHER_DIR
# SCZ-SIB       │ delete folder only  │  —    │ nowhere

GROUP_CONFIG = {
    "SCZ":     {"label": "1", "dest": "NIFTI"},
    "CON":     {"label": "3", "dest": "NIFTI"},
    "CON-SIB": {"label": "6", "dest": "OTHER"},
    "SCZ-SIB": {"label": None, "dest": "DELETE"},
}

# ── SETUP ──────────────────────────────────────────────────────────────────────
os.makedirs(OTHER_DIR, exist_ok=True)

# ── READ TSV ───────────────────────────────────────────────────────────────────
subject_groups = {}  # { "sub-01": "SCZ", ... }

with open(TSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        subj_id = row["participant_id"].strip()
        condit  = row["condit"].strip()
        subject_groups[subj_id] = condit

print(f"Loaded {len(subject_groups)} subjects from TSV.\n")

# ── PROCESS EACH SUBJECT ───────────────────────────────────────────────────────
extracted  = []
deleted    = []
skipped    = []

for subj_id, condit in subject_groups.items():

    subj_folder = os.path.join(NIFTI_DIR, subj_id)

    # Skip if subject folder doesn't exist on disk
    if not os.path.isdir(subj_folder):
        print(f"  [SKIP] {subj_id}: folder not found on disk.")
        skipped.append(subj_id)
        continue

    config = GROUP_CONFIG.get(condit)
    if config is None:
        print(f"  [SKIP] {subj_id}: unknown condit '{condit}' in TSV.")
        skipped.append(subj_id)
        continue

    # ── SCZ-SIB: delete folder, extract nothing ────────────────────────────
    if config["dest"] == "DELETE":
        shutil.rmtree(subj_folder)
        print(f"  [DELETED] {subj_id} (SCZ-SIB) — folder removed, no extraction.")
        deleted.append(subj_id)
        continue

    # ── All other groups: find T1w, copy, rename, delete folder ───────────
    src_file = os.path.join(subj_folder, "anat", f"{subj_id}_T1w.nii.gz")

    if not os.path.isfile(src_file):
        print(f"  [SKIP] {subj_id}: T1w.nii.gz not found at expected path.")
        print(f"         Expected: {src_file}")
        skipped.append(subj_id)
        continue

    label        = config["label"]
    new_filename = f"{subj_id}_T1w_{label}_3.nii.gz"

    if config["dest"] == "NIFTI":
        dest_file = os.path.join(NIFTI_DIR, new_filename)
    else:  # OTHER
        dest_file = os.path.join(OTHER_DIR, new_filename)

    # Copy with new name
    shutil.copy2(src_file, dest_file)

    # Delete subject folder
    shutil.rmtree(subj_folder)

    print(f"  [OK] {subj_id} ({condit} → label {label})")
    print(f"       -> {dest_file}\n")
    extracted.append(subj_id)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("DONE.")
print(f"  Extracted & renamed : {len(extracted)} subjects")
print(f"  Deleted (SCZ-SIB)   : {len(deleted)} subjects")
print(f"  Skipped             : {len(skipped)} subjects -> {skipped}")
print()
print(f"SCZ  files (_1_3) → {NIFTI_DIR}")
print(f"CON  files (_3_3) → {NIFTI_DIR}")
print(f"CON-SIB files (_6_3) → {OTHER_DIR}")
print(f"SCZ-SIB → deleted entirely (no files kept)")

Loaded 99 subjects from TSV.

  [OK] sub-01 (SCZ → label 1)
       -> D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI\sub-01_T1w_1_3.nii.gz

  [DELETED] sub-02 (SCZ-SIB) — folder removed, no extraction.
  [DELETED] sub-03 (SCZ-SIB) — folder removed, no extraction.
  [DELETED] sub-04 (SCZ-SIB) — folder removed, no extraction.
  [OK] sub-05 (SCZ → label 1)
       -> D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI\sub-05_T1w_1_3.nii.gz

  [DELETED] sub-06 (SCZ-SIB) — folder removed, no extraction.
  [OK] sub-07 (SCZ → label 1)
       -> D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI\sub-07_T1w_1_3.nii.gz

  [DELETED] sub-08 (SCZ-SIB) — folder removed, no extraction.
  [OK] sub-09 (SCZ → label 1)
       -> D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI\sub-09_T1w_1_3.nii.gz

  [OK] sub-10 (C

In [9]:
import os
import re

def rename_suffix_0_to_6(directory):
    """
    Renames all .nii.gz files in a directory by replacing
    the trailing _0 suffix (before .nii.gz) with _6.

    e.g. sub-01_T1w_3_0.nii.gz  ->  sub-01_T1w_3_6.nii.gz

    Args:
        directory (str): Path to the folder containing the files
    """
    if not os.path.isdir(directory):
        print(f"[ERROR] Directory not found: {directory}")
        return

    # Match filenames ending in _0.nii.gz
    pattern = re.compile(r'^(.+)_0(\.nii\.gz)$')

    renamed   = []
    skipped   = []
    conflicts = []

    all_files = [f for f in os.listdir(directory) if f.endswith(".nii.gz")]
    print(f"\nDirectory : {directory}")
    print(f"Total .nii.gz files found: {len(all_files)}\n")

    for filename in all_files:
        match = pattern.match(filename)

        if not match:
            skipped.append(filename)
            continue

        new_filename = match.group(1) + "_6" + match.group(2)
        src  = os.path.join(directory, filename)
        dest = os.path.join(directory, new_filename)

        # Safety check — don't overwrite an existing file
        if os.path.exists(dest):
            print(f"  [CONFLICT] {filename} -> {new_filename} already exists, skipping.")
            conflicts.append(filename)
            continue

        os.rename(src, dest)
        print(f"  [RENAMED] {filename}")
        print(f"         -> {new_filename}")
        renamed.append((filename, new_filename))

    # ── SUMMARY ───────────────────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("DONE.")
    print(f"  Renamed    : {len(renamed)}")
    print(f"  Skipped (no _0 suffix)  : {len(skipped)}")
    print(f"  Skipped (name conflict) : {len(conflicts)}")

    if skipped:
        print(f"\nFiles with no _0 suffix (left untouched):")
        for f in skipped:
            print(f"  {f}")


# ── USAGE ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    directory = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI"
    rename_suffix_0_to_6(directory)


Directory : D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI
Total .nii.gz files found: 163

  [RENAMED] CC0197_1_0.nii.gz
         -> CC0197_1_6.nii.gz
  [RENAMED] CC0202_1_0.nii.gz
         -> CC0202_1_6.nii.gz
  [RENAMED] CC0239_3_0.nii.gz
         -> CC0239_3_6.nii.gz
  [RENAMED] CC0325_3_0.nii.gz
         -> CC0325_3_6.nii.gz
  [RENAMED] CC0385_3_0.nii.gz
         -> CC0385_3_6.nii.gz
  [RENAMED] CC0432_1_0.nii.gz
         -> CC0432_1_6.nii.gz
  [RENAMED] CC0475_3_0.nii.gz
         -> CC0475_3_6.nii.gz
  [RENAMED] CC0600_3_0.nii.gz
         -> CC0600_3_6.nii.gz
  [RENAMED] CC0683_1_0.nii.gz
         -> CC0683_1_6.nii.gz
  [RENAMED] CC0717_3_0.nii.gz
         -> CC0717_3_6.nii.gz
  [RENAMED] CC1085_3_0.nii.gz
         -> CC1085_3_6.nii.gz
  [RENAMED] CC1197_3_0.nii.gz
         -> CC1197_3_6.nii.gz
  [RENAMED] CC1381_1_0.nii.gz
         -> CC1381_1_6.nii.gz
  [RENAMED] CC1386_1_0.nii.gz
         -> CC1386_1_6.nii.gz
  [RENAMED] CC1496_3_0.nii.gz


In [11]:
import os
import re

def rename_folders_0_to_6(directory):
    """
    Renames all subfolders in a directory by replacing
    the trailing _0 suffix with _6.

    e.g. sub-01_T1w_3_0  ->  sub-01_T1w_3_6

    Args:
        directory (str): Path to the parent folder containing the subfolders
    """
    if not os.path.isdir(directory):
        print(f"[ERROR] Directory not found: {directory}")
        return

    pattern = re.compile(r'^(.+)_0$')

    renamed   = []
    skipped   = []
    conflicts = []

    all_folders = [f for f in os.listdir(directory)
                   if os.path.isdir(os.path.join(directory, f))]

    print(f"\nDirectory : {directory}")
    print(f"Total subfolders found: {len(all_folders)}\n")

    for folder in all_folders:
        match = pattern.match(folder)

        if not match:
            skipped.append(folder)
            continue

        new_folder = match.group(1) + "_6"
        src  = os.path.join(directory, folder)
        dest = os.path.join(directory, new_folder)

        if os.path.exists(dest):
            print(f"  [CONFLICT] {folder} -> {new_folder} already exists, skipping.")
            conflicts.append(folder)
            continue

        os.rename(src, dest)
        print(f"  [RENAMED] {folder}")
        print(f"         -> {new_folder}")
        renamed.append((folder, new_folder))

    print("\n" + "=" * 60)
    print("DONE.")
    print(f"  Renamed    : {len(renamed)}")
    print(f"  Skipped (no _0 suffix)  : {len(skipped)}")
    print(f"  Skipped (name conflict) : {len(conflicts)}")

    if skipped:
        print(f"\nFolders with no _0 suffix (left untouched):")
        for f in skipped:
            print(f"  {f}")


# ── USAGE ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    directory = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\other_nifti"
    rename_folders_0_to_6(directory)


Directory : D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\other_nifti
Total subfolders found: 50

  [RENAMED] CC0311_5_0
         -> CC0311_5_6
  [RENAMED] CC0486_5_0
         -> CC0486_5_6
  [RENAMED] CC0857_5_0
         -> CC0857_5_6
  [RENAMED] CC0873_5_0
         -> CC0873_5_6
  [RENAMED] CC0949_5_0
         -> CC0949_5_6
  [RENAMED] CC2038_5_0
         -> CC2038_5_6
  [RENAMED] CC2572_5_0
         -> CC2572_5_6
  [RENAMED] CC2762_5_0
         -> CC2762_5_6
  [RENAMED] CC2881_5_0
         -> CC2881_5_6
  [RENAMED] CC2985_5_0
         -> CC2985_5_6
  [RENAMED] CC3125_5_0
         -> CC3125_5_6
  [RENAMED] CC3421_5_0
         -> CC3421_5_6
  [RENAMED] CC3615_5_0
         -> CC3615_5_6
  [RENAMED] CC3803_5_0
         -> CC3803_5_6
  [RENAMED] CC3958_5_0
         -> CC3958_5_6
  [RENAMED] CC4202_5_0
         -> CC4202_5_6
  [RENAMED] CC4513_5_0
         -> CC4513_5_6
  [RENAMED] CC4882_5_0
         -> CC4882_5_6
  [RENAMED] CC4994_5_0
         -> CC499

In [1]:
import os
import shutil
import csv

# ── PATHS ──────────────────────────────────────────────────────────────────────
SOURCE_DIR  = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\redown"
SOURCE_DIR  = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\down2"
NIFTI_DIR   = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\NIFTI"
OTHER_DIR   = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\other_nifti"
TSV_PATH    = os.path.join(SOURCE_DIR, "participants.tsv")

# ── GROUP MAPPING ──────────────────────────────────────────────────────────────
# SCHZ    → label 1 → NIFTI_DIR
# ADHD    → label 2 → OTHER_DIR
# CONTROL → label 3 → NIFTI_DIR
# BIPOLAR → label 4 → OTHER_DIR

DIAGNOSIS_MAP = {
    "SCHZ":    {"label": "1", "dest": "NIFTI"},
    "ADHD":    {"label": "2", "dest": "OTHER"},
    "CONTROL": {"label": "3", "dest": "NIFTI"},
    "BIPOLAR": {"label": "4", "dest": "OTHER"},
}

# ── SETUP ──────────────────────────────────────────────────────────────────────
os.makedirs(NIFTI_DIR, exist_ok=True)
os.makedirs(OTHER_DIR, exist_ok=True)

# ── READ TSV ───────────────────────────────────────────────────────────────────
subject_groups = {}

with open(TSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        subj_id   = row["participant_id"].strip()
        diagnosis = row["diagnosis"].strip().lstrip(",").upper()
        config    = DIAGNOSIS_MAP.get(diagnosis)
        if config:
            subject_groups[subj_id] = config
        else:
            print(f"  [WARN] Unknown diagnosis '{diagnosis}' for {subj_id} — skipping.")

print(f"Loaded {len(subject_groups)} subjects from TSV.\n")

# ── PROCESS EACH SUBJECT ──────────────────────────────────────────────────────
copied   = []
missing  = []
skipped  = []

for subj_id, config in subject_groups.items():

    subj_folder = os.path.join(SOURCE_DIR, subj_id)

    if not os.path.isdir(subj_folder):
        print(f"  [SKIP] {subj_id}: folder not found in source.")
        skipped.append(subj_id)
        continue

    src_t1 = os.path.join(subj_folder, "anat", f"{subj_id}_T1w.nii.gz")

    if not os.path.isfile(src_t1):
        print(f"  [MISS] {subj_id}: T1w.nii.gz not found.")
        missing.append(subj_id)
        continue

    label        = config["label"]
    new_filename = f"{subj_id}_T1w_{label}_4.nii.gz"
    dest_dir     = NIFTI_DIR if config["dest"] == "NIFTI" else OTHER_DIR
    dest_file    = os.path.join(dest_dir, new_filename)

    shutil.copy2(src_t1, dest_file)
    folder_tag = "NIFTI" if config["dest"] == "NIFTI" else "other_nifti"
    print(f"  [OK] {subj_id} -> {new_filename}  [{folder_tag}]")
    copied.append(subj_id)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("DONE. Source folders untouched.")
print(f"  Copied successfully : {len(copied)}")
print(f"  T1w missing        : {len(missing)}  -> {missing}")
print(f"  Folder not on disk : {len(skipped)}  -> {skipped}")
print()
print(f"SCHZ + CONTROL  -> {NIFTI_DIR}")
print(f"BIPOLAR + ADHD  -> {OTHER_DIR}")


Loaded 272 subjects from TSV.

  [SKIP] sub-10159: folder not found in source.
  [SKIP] sub-10171: folder not found in source.
  [SKIP] sub-10189: folder not found in source.
  [SKIP] sub-10193: folder not found in source.
  [SKIP] sub-10206: folder not found in source.
  [SKIP] sub-10217: folder not found in source.
  [SKIP] sub-10225: folder not found in source.
  [SKIP] sub-10227: folder not found in source.
  [SKIP] sub-10228: folder not found in source.
  [SKIP] sub-10235: folder not found in source.
  [SKIP] sub-10249: folder not found in source.
  [SKIP] sub-10269: folder not found in source.
  [SKIP] sub-10271: folder not found in source.
  [SKIP] sub-10273: folder not found in source.
  [SKIP] sub-10274: folder not found in source.
  [SKIP] sub-10280: folder not found in source.
  [SKIP] sub-10290: folder not found in source.
  [SKIP] sub-10292: folder not found in source.
  [SKIP] sub-10299: folder not found in source.
  [SKIP] sub-10304: folder not found in source.
  [SKIP] 

In [2]:
import os
len(os.listdir(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\redown"))

79

In [6]:
missing

['sub-10971',
 'sub-11121',
 'sub-11128',
 'sub-11131',
 'sub-11142',
 'sub-11143',
 'sub-11149',
 'sub-11156',
 'sub-50004',
 'sub-50005',
 'sub-50006',
 'sub-50007',
 'sub-50008',
 'sub-50010',
 'sub-50013',
 'sub-50014',
 'sub-50015',
 'sub-50016',
 'sub-50020',
 'sub-50021',
 'sub-50022',
 'sub-50023',
 'sub-50025',
 'sub-50027',
 'sub-50029',
 'sub-50032',
 'sub-50033',
 'sub-50034',
 'sub-50035',
 'sub-50036',
 'sub-50038',
 'sub-50043',
 'sub-50047',
 'sub-50048',
 'sub-50049',
 'sub-50050',
 'sub-50051',
 'sub-50052',
 'sub-50053',
 'sub-50054',
 'sub-50055',
 'sub-50056',
 'sub-50058',
 'sub-50059',
 'sub-50060',
 'sub-50061',
 'sub-50064',
 'sub-50066',
 'sub-50067',
 'sub-50069',
 'sub-50073',
 'sub-50075',
 'sub-50076',
 'sub-50077',
 'sub-50080',
 'sub-50081',
 'sub-50083',
 'sub-50085',
 'sub-60001',
 'sub-60005',
 'sub-60006',
 'sub-60008',
 'sub-60010',
 'sub-60011',
 'sub-60012',
 'sub-60014',
 'sub-60015',
 'sub-60016',
 'sub-60017',
 'sub-60020',
 'sub-60021',
 'sub-

In [ ]:
import openneuro as on

# ── CONFIG ────────────────────────────────────────────────────────────────────
DEST_DIR   = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\down2"
DATASET_ID = "ds000030"

# ── 150 MISSING SUBJECTS ──────────────────────────────────────────────────────
MISSING = [
    'sub-10971', 'sub-11121', 'sub-11128', 'sub-11131', 'sub-11142',
    'sub-11143', 'sub-11149', 'sub-11156', 'sub-50004', 'sub-50005',
    'sub-50006', 'sub-50007', 'sub-50008', 'sub-50010', 'sub-50013',
    'sub-50014', 'sub-50015', 'sub-50016', 'sub-50020', 'sub-50021',
    'sub-50022', 'sub-50023', 'sub-50025', 'sub-50027', 'sub-50029',
    'sub-50032', 'sub-50033', 'sub-50034', 'sub-50035', 'sub-50036',
    'sub-50038', 'sub-50043', 'sub-50047', 'sub-50048', 'sub-50049',
    'sub-50050', 'sub-50051', 'sub-50052', 'sub-50053', 'sub-50054',
    'sub-50055', 'sub-50056', 'sub-50058', 'sub-50059', 'sub-50060',
    'sub-50061', 'sub-50064', 'sub-50066', 'sub-50067', 'sub-50069',
    'sub-50073', 'sub-50075', 'sub-50076', 'sub-50077', 'sub-50080',
    'sub-50081', 'sub-50083', 'sub-50085', 'sub-60001', 'sub-60005',
    'sub-60006', 'sub-60008', 'sub-60010', 'sub-60011', 'sub-60012',
    'sub-60014', 'sub-60015', 'sub-60016', 'sub-60017', 'sub-60020',
    'sub-60021', 'sub-60022', 'sub-60028', 'sub-60030', 'sub-60033',
    'sub-60036', 'sub-60037', 'sub-60038', 'sub-60042', 'sub-60043',
    'sub-60045', 'sub-60046', 'sub-60048', 'sub-60049', 'sub-60051',
    'sub-60052', 'sub-60053', 'sub-60055', 'sub-60056', 'sub-60057',
    'sub-60060', 'sub-60062', 'sub-60065', 'sub-60066', 'sub-60068',
    'sub-60070', 'sub-60072', 'sub-60073', 'sub-60074', 'sub-60076',
    'sub-60077', 'sub-60078', 'sub-60079', 'sub-60080', 'sub-60084',
    'sub-60087', 'sub-60089', 'sub-70001', 'sub-70002', 'sub-70004',
    'sub-70007', 'sub-70010', 'sub-70015', 'sub-70017', 'sub-70020',
    'sub-70021', 'sub-70022', 'sub-70026', 'sub-70029', 'sub-70033',
    'sub-70034', 'sub-70035', 'sub-70036', 'sub-70037', 'sub-70040',
    'sub-70046', 'sub-70048', 'sub-70049', 'sub-70051', 'sub-70052',
    'sub-70055', 'sub-70057', 'sub-70058', 'sub-70060', 'sub-70061',
    'sub-70065', 'sub-70068', 'sub-70069', 'sub-70070', 'sub-70072',
    'sub-70073', 'sub-70074', 'sub-70075', 'sub-70076', 'sub-70077',
    'sub-70079', 'sub-70080', 'sub-70081', 'sub-70083', 'sub-70086',
]

print(f"Subjects to download : {len(MISSING)}")
print(f"Destination          : {DEST_DIR}\n")
print("Starting download via openneuro-py...\n")

on.download(
    dataset    = DATASET_ID,
    target_dir = DEST_DIR,
    include    = MISSING,
)

print("\nDownload complete!")


d:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\pre-processing\prevenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Subjects to download : 150
Destination          : D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\down2

Starting download via openneuro-py...


👋 Hello! This is openneuro-py 2026.3.0. Great to see you! 🤗

   👉 Please report problems 🤯 and bugs 🪲 at
      https://github.com/hoechenberger/openneuro-py/issues

🌍 Preparing to download ds000030 …


📁 Traversing directories for ds000030 : 5067 entities [05:04, 16.65 entities/s]

📥 Retrieving up to 5059 files (5 concurrent downloads). 
✅ Finished downloading ds000030.
 
🧠 Please enjoy your brains.
 

Download complete!


sub-11128_T1w.nii.gz:   0%|          | 32.6k/11.3M [00:00<01:14, 158kB/s]                                               

sub-11128_T1w.nii.gz:   1%|          | 83.6k/11.3M [00:00<00:55, 211kB/s]




sub-11128_T1w.nii.gz:   2%|▏         | 186k/11.3M [00:00<00:33, 345kB/s] 


sub-11128_T1w.nii.gz:   3%|▎         | 322k/11.3M [00:00<00:24, 463kB/s]





sub-11128_T1w.nii.gz:   5%|▍         | 577k/11.3M [00:01<00:15, 747kB/s]



sub-11128_T1w.nii.gz:   9%|▉         | 1.06M/11.3M [00:01<00:08, 1.34MB/s]


sub-11128_T1w.nii.gz:  17%|█▋        | 1.87M/11.3M [00:01<00:03, 2.64MB/s]
sub-11128_T1w.nii.gz:  20%|█▉        | 2.21M/11.3M [00:01<00:03, 2.68MB/s]


sub-11128_T1w.nii.gz:  26%|██▌       | 2.91M/11.3M [00:01<00:02, 3.71MB/s]


sub-11128_T1w.nii.gz:  34%|███▎      | 3.80M/11.3M [00:01<00:01, 5.06MB/s]


sub-11128_T1w.nii.gz:  39%|███▉      | 4.38M/11.3M [00:01<00:01, 4.21MB/s]


sub-11128_T1w.nii.gz:  47%|████▋     | 5.33M/11.3M [00:02<00:01, 5.45MB/s]


sub-11128_T1w.nii.gz:  54%|█████▍

In [1]:
import os
import csv
import shutil
import subprocess
import tempfile

# ── CONFIG ────────────────────────────────────────────────────────────────────
SOURCE_DIR = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\schiz_strict"
CSV_PATH   = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\schiz_strict.csv"
NIFTI_DIR  = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\NIFTI"

os.makedirs(NIFTI_DIR, exist_ok=True)

# ── READ CSV — one Datauri per subject (first occurrence) ─────────────────────
subject_to_datauri = {}  # Subject ID -> first Datauri folder

with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        subj_id  = row["Subject ID"].strip()
        datauri  = row["Datauri"].strip()
        if subj_id not in subject_to_datauri:
            subject_to_datauri[subj_id] = datauri

print(f"Subjects to process: {len(subject_to_datauri)}\n")

# ── HELPER: find the deepest folder containing .dcm files ─────────────────────
def find_dcm_folder(root):
    """Walk the tree and return the first folder that contains .dcm files."""
    for dirpath, dirnames, filenames in os.walk(root):
        dcm_files = [f for f in filenames if f.lower().endswith(".dcm")]
        if dcm_files:
            return dirpath
    return None

# ── PROCESS ───────────────────────────────────────────────────────────────────
done    = []
missing = []
failed  = []

for subj_id, datauri in subject_to_datauri.items():

    subj_folder = os.path.join(SOURCE_DIR, datauri)

    if not os.path.isdir(subj_folder):
        print(f"  [SKIP] {subj_id} (datauri={datauri}): folder not found.")
        missing.append(subj_id)
        continue

    dcm_folder = find_dcm_folder(subj_folder)
    if not dcm_folder:
        print(f"  [SKIP] {subj_id}: no .dcm files found under {subj_folder}")
        missing.append(subj_id)
        continue

    # Convert DCM → NIfTI in a temp folder
    with tempfile.TemporaryDirectory() as tmpdir:
        cmd = [
            "dcm2niix",
            "-z", "y",          # output .nii.gz
            "-f", "converted",  # output filename inside tmpdir
            "-o", tmpdir,
            dcm_folder
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        # Find the .nii.gz output
        nii_files = [f for f in os.listdir(tmpdir) if f.endswith(".nii.gz")]

        if not nii_files:
            print(f"  [FAIL] {subj_id}: dcm2niix produced no .nii.gz")
            print(f"         dcm2niix stderr: {result.stderr.strip()}")
            failed.append(subj_id)
            continue

        # If multiple outputs (unlikely for T1), pick the first
        src_nii = os.path.join(tmpdir, nii_files[0])

        dest_filename = f"{subj_id}_T1_1_7.nii.gz"
        dest_path     = os.path.join(NIFTI_DIR, dest_filename)

        shutil.copy2(src_nii, dest_path)
        print(f"  [OK] {subj_id} -> {dest_filename}")
        done.append(subj_id)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"Converted successfully : {len(done)}")
print(f"Folder/DCM not found   : {len(missing)}  -> {missing}")
print(f"dcm2niix failed        : {len(failed)}  -> {failed}")
print(f"\nOutput -> {NIFTI_DIR}")

Subjects to process: 79

  [OK] A00014636 -> A00014636_T1_1_7.nii.gz
  [OK] A00014804 -> A00014804_T1_1_7.nii.gz
  [OK] A00014830 -> A00014830_T1_1_7.nii.gz
  [OK] A00024959 -> A00024959_T1_1_7.nii.gz
  [OK] A00024568 -> A00024568_T1_1_7.nii.gz
  [OK] A00024228 -> A00024228_T1_1_7.nii.gz
  [OK] A00024198 -> A00024198_T1_1_7.nii.gz
  [OK] A00024953 -> A00024953_T1_1_7.nii.gz
  [OK] A00016197 -> A00016197_T1_1_7.nii.gz
  [OK] A00023750 -> A00023750_T1_1_7.nii.gz
  [OK] A00027616 -> A00027616_T1_1_7.nii.gz
  [OK] A00023158 -> A00023158_T1_1_7.nii.gz
  [OK] A00000456 -> A00000456_T1_1_7.nii.gz
  [OK] A00023132 -> A00023132_T1_1_7.nii.gz
  [OK] A00023366 -> A00023366_T1_1_7.nii.gz
  [OK] A00023243 -> A00023243_T1_1_7.nii.gz
  [OK] A00022500 -> A00022500_T1_1_7.nii.gz
  [OK] A00020787 -> A00020787_T1_1_7.nii.gz
  [OK] A00023590 -> A00023590_T1_1_7.nii.gz
  [OK] A00024684 -> A00024684_T1_1_7.nii.gz
  [OK] A00020414 -> A00020414_T1_1_7.nii.gz
  [OK] A00000838 -> A00000838_T1_1_7.nii.gz
  [OK] 

KeyboardInterrupt: 

In [2]:
import os
import shutil
import subprocess
import tempfile

# ── CONFIG ────────────────────────────────────────────────────────────────────
SOURCE_DIR = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\schiz_strict"
NIFTI_DIR  = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\NIFTI"

os.makedirs(NIFTI_DIR, exist_ok=True)

# ── REMAINING 39 UNIQUE SUBJECTS (not yet converted) ─────────────────────────
# subject_id -> datauri (top-level folder under schiz_strict)
REMAINING = {
    'A00001251': '721474',
    'A00015648': '771954',
    'A00015201': '816474',
    'A00027537': '832414',
    'A00028404': '841614',
    'A00000909': '857934',
    'A00006754': '900594',
    'A00018403': '950034',
    'A00000368': '970694',
    'A00001243': '988474',
    'A00021598': '1010074',
    'A00009280': '1036674',
    'A00002405': '1052174',
    'A00023246': '1077714',
    'A00027391': '1085374',
    'A00016720': '1116214',
    'A00021591': '1116754',
    'A00014719': '1178846',
    'A00028410': '1261246',
    'A00027410': '1281706',
    'A00028408': '1330306',
    'A00004507': '1339226',
    'A00028405': '1381886',
    'A00028303': '1388746',
    'A00028805': '1432366',
    'A00028806': '1433826',
    'A00029486': '1519626',
    'A00031186': '1541486',
    'A00031597': '1561386',
    'A00034273': '1712646',
    'A00035003': '1759046',
    'A00035485': '1833646',
    'A00037034': '2030406',
    'A00037224': '2087506',
    'A00037619': '2122386',
    'A00037649': '2142146',
    'A00037854': '2168306',
    'A00038441': '2227786',
    'A00038624': '2294306',
}

# ── HELPER: find folder containing .dcm files ─────────────────────────────────
def find_dcm_folder(root):
    for dirpath, dirnames, filenames in os.walk(root):
        if any(f.lower().endswith(".dcm") for f in filenames):
            return dirpath
    return None

# ── PROCESS ───────────────────────────────────────────────────────────────────
done    = []
missing = []
failed  = []

print(f"Subjects to convert: {len(REMAINING)}\n")

for subj_id, datauri in REMAINING.items():

    # Skip if already converted
    dest_filename = f"{subj_id}_T1_1_7.nii.gz"
    dest_path     = os.path.join(NIFTI_DIR, dest_filename)
    if os.path.isfile(dest_path):
        print(f"  [EXIST] {subj_id}: already done, skipping.")
        done.append(subj_id)
        continue

    subj_folder = os.path.join(SOURCE_DIR, datauri)
    if not os.path.isdir(subj_folder):
        print(f"  [SKIP] {subj_id} (datauri={datauri}): folder not on disk.")
        missing.append(subj_id)
        continue

    dcm_folder = find_dcm_folder(subj_folder)
    if not dcm_folder:
        print(f"  [SKIP] {subj_id}: no .dcm files found.")
        missing.append(subj_id)
        continue

    with tempfile.TemporaryDirectory() as tmpdir:
        cmd = ["dcm2niix", "-z", "y", "-f", "converted", "-o", tmpdir, dcm_folder]
        result = subprocess.run(cmd, capture_output=True, text=True)

        nii_files = [f for f in os.listdir(tmpdir) if f.endswith(".nii.gz")]

        if not nii_files:
            print(f"  [FAIL] {subj_id}: dcm2niix produced no output.")
            print(f"         stderr: {result.stderr.strip()}")
            failed.append(subj_id)
            continue

        shutil.copy2(os.path.join(tmpdir, nii_files[0]), dest_path)
        print(f"  [OK] {subj_id} -> {dest_filename}")
        done.append(subj_id)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"Converted successfully : {len(done)}")
print(f"Folder/DCM not found   : {len(missing)}  -> {missing}")
print(f"dcm2niix failed        : {len(failed)}  -> {failed}")
print(f"\nOutput -> {NIFTI_DIR}")

Subjects to convert: 39

  [OK] A00001251 -> A00001251_T1_1_7.nii.gz
  [OK] A00015648 -> A00015648_T1_1_7.nii.gz
  [OK] A00015201 -> A00015201_T1_1_7.nii.gz
  [OK] A00027537 -> A00027537_T1_1_7.nii.gz
  [OK] A00028404 -> A00028404_T1_1_7.nii.gz
  [OK] A00000909 -> A00000909_T1_1_7.nii.gz
  [OK] A00006754 -> A00006754_T1_1_7.nii.gz
  [OK] A00018403 -> A00018403_T1_1_7.nii.gz
  [OK] A00000368 -> A00000368_T1_1_7.nii.gz
  [OK] A00001243 -> A00001243_T1_1_7.nii.gz
  [OK] A00021598 -> A00021598_T1_1_7.nii.gz
  [OK] A00009280 -> A00009280_T1_1_7.nii.gz
  [OK] A00002405 -> A00002405_T1_1_7.nii.gz
  [OK] A00023246 -> A00023246_T1_1_7.nii.gz
  [OK] A00027391 -> A00027391_T1_1_7.nii.gz
  [OK] A00016720 -> A00016720_T1_1_7.nii.gz
  [OK] A00021591 -> A00021591_T1_1_7.nii.gz
  [OK] A00014719 -> A00014719_T1_1_7.nii.gz
  [OK] A00028410 -> A00028410_T1_1_7.nii.gz
  [OK] A00027410 -> A00027410_T1_1_7.nii.gz
  [OK] A00028408 -> A00028408_T1_1_7.nii.gz
  [OK] A00004507 -> A00004507_T1_1_7.nii.gz
  [OK] 

In [3]:
import os
os.listdir(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\analyse_incomplete")

['CC7557',
 'CC9555',
 'NM1002',
 'NM1003',
 'NM1007',
 'NM1008',
 'NM1009',
 'NM1010',
 'NM1011',
 'NM1013',
 'NM1014',
 'NM1016',
 'NM1017',
 'NM1018',
 'NM1019',
 'NM1020',
 'NM1021',
 'NM1022',
 'NM1023',
 'NM1025',
 'NM1029',
 'NM1030',
 'NM1031',
 'NM1032',
 'NM1034',
 'NM1035',
 'NM1036',
 'NM1037',
 'NM1038',
 'NM1039',
 'NM1040',
 'NM1041',
 'NM1042',
 'NM1043',
 'NM1044',
 'NM1045',
 'NM1046',
 'NM1047',
 'NM1049',
 'NM1050',
 'NM1051',
 'NM1060',
 'NM1061',
 'NM1063',
 'NM1064',
 'NM1065',
 'NM1067',
 'NM1068',
 'NM1069',
 'NM1070',
 'NM1071',
 'NM1073',
 'NM2001',
 'NM2002',
 'NM2003',
 'NM2004',
 'NM2006',
 'NM2007',
 'NM2008',
 'NM2010',
 'NM2012',
 'NM2013',
 'NM2014',
 'NM2015',
 'NM2016',
 'NM2017',
 'NM2018',
 'NM2020',
 'NM2022',
 'NM2025',
 'NM2027',
 'NM2028',
 'NM2029',
 'NM2031',
 'NM2040',
 'NM2043',
 'NM2047',
 'NM2050',
 'NM2052',
 'NM2053',
 'NM2054',
 'NM2056',
 'NM2057',
 'NM2058',
 'NM2060',
 'NM2061',
 'NM2062',
 'NM2063',
 'NM2064',
 'NM2065',
 'NM2066',

In [2]:
import os
for folder, subfolders, files in os.walk(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\redownn_analyse\CC0005_0"):
    print(f"Folder: {folder}")
    print(f"  Subfolders: {subfolders}")
    print(f"  Files: {files}\n")

Folder: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\redownn_analyse\CC0005_0
  Subfolders: ['FLASH1', 'MPR1', 'MPR2']
  Files: []

Folder: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\redownn_analyse\CC0005_0\FLASH1
  Subfolders: ['ANALYZE']
  Files: []

Folder: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\redownn_analyse\CC0005_0\FLASH1\ANALYZE
  Subfolders: []
  Files: ['CC0005_0_1219-2.hdr', 'CC0005_0_1219-2.ifh', 'CC0005_0_1219-2.img']

Folder: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\redownn_analyse\CC0005_0\MPR1
  Subfolders: ['ANALYZE', 'NIFTI']
  Files: []

Folder: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\redownn_analyse\CC0005_0\MPR1\ANALYZE
  Subfolders: []
  Files: ['CC0005_0_1219-4.hdr', 'CC0005_0_1219-4.img']

Folder: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\r

In [4]:
import os
import csv
import shutil
import nibabel as nib
import numpy as np

# ── CONFIG ────────────────────────────────────────────────────────────────────
SOURCE_DIR = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\redownn_analyse"
NIFTI_DIR  = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI"
CSV_PATH   = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\zaynul23_3_6_2026_2_32_57.csv"

ALLOWED_GROUPS = {'1', '3', '5'}

os.makedirs(NIFTI_DIR, exist_ok=True)

# ── READ CSV ──────────────────────────────────────────────────────────────────
subject_group = {}
with open(CSV_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sid = row["Subject"].strip()
        grp = row["Group"].strip()
        if sid:
            subject_group[sid] = grp

print(f"Loaded {len(subject_group)} subjects from CSV.\n")

# ── HELPER: convert .hdr/.img to .nii.gz using nibabel ───────────────────────
def analyze_to_nifti(hdr_path, out_path):
    img = nib.load(hdr_path)                        # loads Analyze pair
    nii = nib.Nifti1Image(np.array(img.dataobj), img.affine, img.header)
    nib.save(nii, out_path)

# ── HELPER: find .nii.gz in a folder ─────────────────────────────────────────
def find_nifti(folder):
    if not os.path.isdir(folder):
        return None
    files = [f for f in os.listdir(folder) if f.endswith(".nii.gz")]
    return os.path.join(folder, files[0]) if files else None

# ── HELPER: find .hdr in a folder ────────────────────────────────────────────
def find_hdr(folder):
    if not os.path.isdir(folder):
        return None
    files = [f for f in os.listdir(folder) if f.endswith(".hdr")]
    return os.path.join(folder, files[0]) if files else None

# ── PROCESS ───────────────────────────────────────────────────────────────────
done    = []
skipped = []
missing = []
failed  = []

subject_folders = sorted([
    d for d in os.listdir(SOURCE_DIR)
    if os.path.isdir(os.path.join(SOURCE_DIR, d))
])

print(f"Subject folders found on disk: {len(subject_folders)}\n")

for folder_name in subject_folders:

    # Strip _0 / _24 suffix to get base subject ID
    subj_id = folder_name.rsplit("_", 1)[0] if "_" in folder_name else folder_name

    grp = subject_group.get(subj_id, "").strip()

    if grp not in ALLOWED_GROUPS:
        print(f"  [SKIP] {subj_id}: group '{grp}' not in allowed set.")
        skipped.append(subj_id)
        continue

    dest_filename = f"{subj_id}_T1_{grp}_6.nii.gz"
    dest_path     = os.path.join(NIFTI_DIR, dest_filename)

    if os.path.isfile(dest_path):
        print(f"  [EXIST] {subj_id}: already converted, skipping.")
        done.append(subj_id)
        continue

    subj_folder = os.path.join(SOURCE_DIR, folder_name)

    # ── PRIORITY: MPR1/NIFTI → MPR1/ANALYZE → MPR2/NIFTI → MPR2/ANALYZE ──
    src_nii = None
    method  = None

    mpr1_nifti   = os.path.join(subj_folder, "MPR1", "NIFTI")
    mpr1_analyze = os.path.join(subj_folder, "MPR1", "ANALYZE")
    mpr2_nifti   = os.path.join(subj_folder, "MPR2", "NIFTI")
    mpr2_analyze = os.path.join(subj_folder, "MPR2", "ANALYZE")

    nii = find_nifti(mpr1_nifti)
    if nii:
        src_nii = nii
        method  = "MPR1/NIFTI"
    else:
        hdr = find_hdr(mpr1_analyze)
        if hdr:
            src_nii = hdr
            method  = "MPR1/ANALYZE"
        else:
            nii = find_nifti(mpr2_nifti)
            if nii:
                src_nii = nii
                method  = "MPR2/NIFTI"
            else:
                hdr = find_hdr(mpr2_analyze)
                if hdr:
                    src_nii = hdr
                    method  = "MPR2/ANALYZE"

    if not src_nii:
        print(f"  [MISS] {subj_id}: no MPR1/MPR2 NIfTI or Analyze found.")
        missing.append(subj_id)
        continue

    try:
        if method.endswith("NIFTI"):
            shutil.copy2(src_nii, dest_path)
        else:
            analyze_to_nifti(src_nii, dest_path)

        print(f"  [OK] {subj_id} -> {dest_filename}  [{method}]")
        done.append(subj_id)

    except Exception as e:
        print(f"  [FAIL] {subj_id}: {e}")
        failed.append(subj_id)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print(f"Converted/copied       : {len(done)}")
print(f"Skipped (wrong group)  : {len(skipped)}")
print(f"Missing scan files     : {len(missing)}  -> {missing}")
print(f"Failed                 : {len(failed)}   -> {failed}")
print(f"\nOutput -> {NIFTI_DIR}")

Loaded 463 subjects from CSV.

Subject folders found on disk: 368

  [SKIP] CC0005: group '2' not in allowed set.
  [SKIP] CC0196: group '2' not in allowed set.


C:\Users\zaynu\AppData\Local\Temp\ipykernel_33732\2683614350.py:31: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  nii = nib.Nifti1Image(np.array(img.dataobj), img.affine, img.header)


  [OK] CC0197 -> CC0197_T1_1_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0202 -> CC0202_T1_1_6.nii.gz  [MPR1/ANALYZE]
  [SKIP] CC0212: group '2' not in allowed set.
  [OK] CC0239 -> CC0239_T1_3_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0311 -> CC0311_T1_5_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0325 -> CC0325_T1_3_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0385 -> CC0385_T1_3_6.nii.gz  [MPR1/ANALYZE]
  [SKIP] CC0409: group '2' not in allowed set.
  [OK] CC0432 -> CC0432_T1_1_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0475 -> CC0475_T1_3_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0486 -> CC0486_T1_5_6.nii.gz  [MPR1/ANALYZE]
  [SKIP] CC0504: group '2' not in allowed set.
  [SKIP] CC0556: group '2' not in allowed set.
  [OK] CC0600 -> CC0600_T1_3_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0683 -> CC0683_T1_1_6.nii.gz  [MPR1/ANALYZE]
  [SKIP] CC0713: group '2' not in allowed set.
  [OK] CC0717 -> CC0717_T1_3_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0857 -> CC0857_T1_5_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC0873 -> CC0873_T1_5_6.nii.gz  [MPR1/ANALYZE]
  [OK] CC

In [7]:
len(os.listdir(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\analyse_incomplete"))

121

In [8]:
import os
for root, dirs, files in os.walk(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data"):
    if root.startswith(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\cobre\scans\controls"):
        continue  # skip printing files in this folder
    if root.startswith(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\redownn_analyse"):
        continue  # skip printing files in this folder
    if root.startswith(r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_5_4"):
        continue  # skip printing files in this folder
    print(f"Root: {root}")
    print(f"  Dirs: {dirs}")
    # print(f"  Files: {files}\n")
    

Root: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data
  Dirs: ['NITRC', 'open_neuro', 'schizconnect']
Root: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC
  Dirs: ['NUSDAST']
Root: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST
  Dirs: ['scans']
Root: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans
  Dirs: ['analyse_incomplete', 'NIFTI', 'other_nifti', 'redownn_analyse', 'test']
Root: D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\analyse_incomplete
  Dirs: ['CC7557', 'CC9555', 'NM1002', 'NM1003', 'NM1007', 'NM1008', 'NM1009', 'NM1010', 'NM1011', 'NM1013', 'NM1014', 'NM1016', 'NM1017', 'NM1018', 'NM1019', 'NM1020', 'NM1021', 'NM1022', 'NM1023', 'NM1025', 'NM1029', 'NM1030', 'NM1031', 'NM1032', 'NM1034', 'NM1035', 'NM1036', 'NM1037', 'NM1038', 'NM1039', 'NM1040', 'NM1041', 'NM1042', 'NM1043', 'NM1044', 'NM1045', 'NM1046', 'NM1047', 'NM1049', 'NM1050', 'NM1051', 'NM1060',

In [26]:
import os
import csv

# ── GRP -> BINARY LABEL ───────────────────────────────────────────────────────
# 1 = Schizophrenia       -> 1
# 3 = Healthy Control     -> 0
# 5 = Schiz Spectrum      -> 1
GRP_TO_LABEL = {
    "1": 1,
    "3": 0,
    "5": 1,
}

# ── ALL NIFTI FOLDERS ─────────────────────────────────────────────────────────
NIFTI_FOLDERS = [
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_0_71\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_1_26\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_2_16\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_3_58\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_4_50\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\open_neuro\open_neuro_5_4\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\NITRC\NUSDAST\scans\NIFTI",
    r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\schizconnect\cobre\scans\NIFTI",
]

OUTPUT_CSV = r"D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\master_scan_list.csv"

# ── SCAN AND WRITE ────────────────────────────────────────────────────────────
rows    = []
unknown = []

for folder in NIFTI_FOLDERS:
    if not os.path.isdir(folder):
        print(f"  [WARN] Not found: {folder}")
        continue

    files = sorted([f for f in os.listdir(folder) if f.endswith(".nii.gz")])

    for fname in files:
        base  = fname.replace(".nii.gz", "")
        parts = base.split("_")

        if len(parts) < 3:
            print(f"  [WARN] Can't parse: {fname}")
            unknown.append(fname)
            continue

        grp    = parts[-2]
        dscode = parts[-1]
        label  = GRP_TO_LABEL.get(grp, None)

        if label is None:
            print(f"  [WARN] Unknown grp '{grp}' in: {fname}")
            unknown.append(fname)
            continue

        filepath = os.path.join(folder, fname)

        rows.append({
            "filename": fname,
            "filepath": filepath,
            "grp":      grp,
            "dscode":   dscode,
            "label":    label,
            "split":    "not decided",
        })

# ── WRITE CSV ─────────────────────────────────────────────────────────────────
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "filepath", "grp", "dscode", "label", "split"])
    writer.writeheader()
    writer.writerows(rows)

# ── SUMMARY ───────────────────────────────────────────────────────────────────
schiz   = sum(1 for r in rows if r["label"] == 1)
healthy = sum(1 for r in rows if r["label"] == 0)

print(f"\nTotal scans written : {len(rows)}")
print(f"  Schizophrenia (1) : {schiz}")
print(f"  Healthy       (0) : {healthy}")
print(f"Unparseable files   : {len(unknown)}")
print(f"CSV saved to        : {OUTPUT_CSV}")


Total scans written : 876
  Schizophrenia (1) : 361
  Healthy       (0) : 515
Unparseable files   : 0
CSV saved to        : D:\zaynu\Documents\coding\PS\2-2\schizobrain-Scan\MRI_data\master_scan_list.csv


In [ ]:
''' 
BAD HEADER: CC0197_1_6.nii.gz
BAD HEADER: CC0202_1_6.nii.gz
BAD HEADER: CC0239_3_6.nii.gz
BAD HEADER: CC0325_3_6.nii.gz
BAD HEADER: CC0385_3_6.nii.gz
BAD HEADER: CC0432_1_6.nii.gz
BAD HEADER: CC0475_3_6.nii.gz
BAD HEADER: CC0600_3_6.nii.gz
BAD HEADER: CC0683_1_6.nii.gz
BAD HEADER: CC0717_3_6.nii.gz
BAD HEADER: CC1085_3_6.nii.gz
BAD HEADER: CC1197_3_6.nii.gz
BAD HEADER: CC1381_1_6.nii.gz
BAD HEADER: CC1386_1_6.nii.gz
BAD HEADER: CC1496_3_6.nii.gz
BAD HEADER: CC1549_1_6.nii.gz
BAD HEADER: CC1570_3_6.nii.gz
BAD HEADER: CC1709_3_6.nii.gz
BAD HEADER: CC1960_3_6.nii.gz
BAD HEADER: CC2191_1_6.nii.gz
BAD HEADER: CC2199_1_6.nii.gz
BAD HEADER: CC2263_1_6.nii.gz
BAD HEADER: CC2430_1_6.nii.gz
BAD HEADER: CC2442_3_6.nii.gz
BAD HEADER: CC2485_1_6.nii.gz
BAD HEADER: CC2606_3_6.nii.gz
BAD HEADER: CC2825_3_6.nii.gz
BAD HEADER: CC3037_3_6.nii.gz
BAD HEADER: CC3493_1_6.nii.gz
BAD HEADER: CC3499_1_6.nii.gz
BAD HEADER: CC3915_3_6.nii.gz
BAD HEADER: CC3945_3_6.nii.gz
BAD HEADER: CC4094_3_6.nii.gz
BAD HEADER: CC4183_3_6.nii.gz
BAD HEADER: CC4189_3_6.nii.gz
BAD HEADER: CC4431_3_6.nii.gz
BAD HEADER: CC4526_3_6.nii.gz
BAD HEADER: CC4896_3_6.nii.gz
BAD HEADER: CC5006_3_6.nii.gz
BAD HEADER: CC5041_1_6.nii.gz
BAD HEADER: CC5111_3_6.nii.gz
BAD HEADER: CC5479_1_6.nii.gz
BAD HEADER: CC5529_3_6.nii.gz
BAD HEADER: CC5562_3_6.nii.gz
BAD HEADER: CC5721_1_6.nii.gz
BAD HEADER: CC5884_3_6.nii.gz
BAD HEADER: CC5892_3_6.nii.gz
BAD HEADER: CC5989_3_6.nii.gz
BAD HEADER: CC6167_1_6.nii.gz
BAD HEADER: CC6182_1_6.nii.gz
BAD HEADER: CC6183_3_6.nii.gz
BAD HEADER: CC6238_1_6.nii.gz
BAD HEADER: CC6476_1_6.nii.gz
BAD HEADER: CC6584_3_6.nii.gz
BAD HEADER: CC6606_1_6.nii.gz
BAD HEADER: CC6681_1_6.nii.gz
BAD HEADER: CC6823_3_6.nii.gz
BAD HEADER: CC6968_1_6.nii.gz
BAD HEADER: CC7028_1_6.nii.gz
BAD HEADER: CC7233_3_6.nii.gz
BAD HEADER: CC7234_3_6.nii.gz
BAD HEADER: CC7335_3_6.nii.gz
BAD HEADER: CC7411_3_6.nii.gz
BAD HEADER: CC7427_1_6.nii.gz
BAD HEADER: CC7493_1_6.nii.gz
BAD HEADER: CC7537_3_6.nii.gz
BAD HEADER: CC7583_3_6.nii.gz
BAD HEADER: CC7872_1_6.nii.gz
BAD HEADER: CC7910_3_6.nii.gz
BAD HEADER: CC7911_1_6.nii.gz
BAD HEADER: CC7926_1_6.nii.gz
BAD HEADER: CC7959_1_6.nii.gz
BAD HEADER: CC7960_1_6.nii.gz
BAD HEADER: CC8016_3_6.nii.gz
BAD HEADER: CC8078_3_6.nii.gz
BAD HEADER: CC8225_1_6.nii.gz
BAD HEADER: CC8323_3_6.nii.gz
BAD HEADER: CC8599_1_6.nii.gz
BAD HEADER: CC8913_1_6.nii.gz
BAD HEADER: CC9044_3_6.nii.gz
BAD HEADER: CC9301_1_6.nii.gz
BAD HEADER: CC9492_1_6.nii.gz
BAD HEADER: CC9540_1_6.nii.gz
BAD HEADER: CC9628_3_6.nii.gz
BAD HEADER: CC9673_3_6.nii.gz
BAD HEADER: CC9744_1_6.nii.gz
BAD HEADER: CC9940_1_6.nii.gz
BAD HEADER: NM0413_1_6.nii.gz
BAD HEADER: NM0909_1_6.nii.gz
BAD HEADER: NM1001_1_6.nii.gz
BAD HEADER: NM1005_1_6.nii.gz
BAD HEADER: NM1015_1_6.nii.gz
BAD HEADER: NM1072_1_6.nii.gz
BAD HEADER: NM1074_1_6.nii.gz
BAD HEADER: NM1076_1_6.nii.gz
BAD HEADER: NM1077_1_6.nii.gz
BAD HEADER: NM1078_1_6.nii.gz
BAD HEADER: NM1080_1_6.nii.gz
BAD HEADER: NM1081_1_6.nii.gz
BAD HEADER: NM1082_1_6.nii.gz
BAD HEADER: NM1083_1_6.nii.gz
BAD HEADER: NM1085_1_6.nii.gz
BAD HEADER: NM1086_1_6.nii.gz
BAD HEADER: NM1087_1_6.nii.gz
BAD HEADER: NM1088_1_6.nii.gz
BAD HEADER: NM1089_1_6.nii.gz
BAD HEADER: NM2102_3_6.nii.gz
BAD HEADER: NM2103_3_6.nii.gz
BAD HEADER: NM2104_3_6.nii.gz
BAD HEADER: NM2105_3_6.nii.gz
BAD HEADER: NM2106_3_6.nii.gz
BAD HEADER: NM2107_3_6.nii.gz
BAD HEADER: NM2109_3_6.nii.gz
BAD HEADER: NM2110_3_6.nii.gz
BAD HEADER: NM2111_3_6.nii.gz
BAD HEADER: NM3215_1_6.nii.gz
BAD HEADER: NM3356_1_6.nii.gz
BAD HEADER: NM3455_1_6.nii.gz
BAD HEADER: NM4038_1_6.nii.gz
BAD HEADER: NM4153_1_6.nii.gz
BAD HEADER: NM4194_1_6.nii.gz
BAD HEADER: NM4329_3_6.nii.gz
BAD HEADER: NM4405_1_6.nii.gz
BAD HEADER: NM4443_1_6.nii.gz
BAD HEADER: NM4522_3_6.nii.gz
BAD HEADER: NM4558_3_6.nii.gz
BAD HEADER: NM4618_1_6.nii.gz
BAD HEADER: NM4756_1_6.nii.gz
BAD HEADER: NM4764_1_6.nii.gz
BAD HEADER: NM4869_1_6.nii.gz
BAD HEADER: NM5079_3_6.nii.gz
BAD HEADER: NM5231_1_6.nii.gz
BAD HEADER: NM5277_1_6.nii.gz
BAD HEADER: NM5747_1_6.nii.gz
BAD HEADER: NM5860_1_6.nii.gz
BAD HEADER: NM5920_1_6.nii.gz
BAD HEADER: NM5946_3_6.nii.gz
BAD HEADER: NM6112_1_6.nii.gz
BAD HEADER: NM6211_1_6.nii.gz
BAD HEADER: NM6224_1_6.nii.gz
BAD HEADER: NM6285_1_6.nii.gz
BAD HEADER: NM6385_1_6.nii.gz
BAD HEADER: NM6433_1_6.nii.gz
BAD HEADER: NM6450_3_6.nii.gz
BAD HEADER: NM6664_1_6.nii.gz
BAD HEADER: NM6683_1_6.nii.gz
BAD HEADER: NM7298_1_6.nii.gz
BAD HEADER: NM7365_3_6.nii.gz
BAD HEADER: NM7465_1_6.nii.gz
BAD HEADER: NM7595_1_6.nii.gz
BAD HEADER: NM7655_3_6.nii.gz
BAD HEADER: NM8228_3_6.nii.gz
BAD HEADER: NM8247_1_6.nii.gz
BAD HEADER: NM8352_1_6.nii.gz
BAD HEADER: NM8356_3_6.nii.gz
BAD HEADER: NM8365_1_6.nii.gz
BAD HEADER: NM8659_1_6.nii.gz
BAD HEADER: NM8724_1_6.nii.gz
BAD HEADER: NM9000_1_6.nii.gz
BAD HEADER: NM9055_1_6.nii.gz
BAD HEADER: NM9107_3_6.nii.gz
BAD HEADER: NM9324_1_6.nii.gz
BAD HEADER: NM9798_1_6.nii.gz'''.count("BAD HEADER")

: 